# Model Selection and Comparison — Making the Call Like a Professional

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb14_model_selection_protocol.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Set up a **selection protocol** that evaluates the **full lineage of reference models from nb01-nb13** — 7 candidates for the classification case, 8 for regression — on identical cross-validation folds with a declared primary metric, before any results are seen.
2. Compute the **95% confidence interval** on the champion's CV scores using Student's *t* (the same statistic you learned in nb08); apply the **CI-overlap rule** to decide whether the top model has earned the right to displace the simpler runner-up.
3. Justify the champion pick in **stakeholder language** — what was compared, what was picked, and why.
4. Open the **locked test set exactly once per case** and pronounce an INSIDE / ABOVE / BELOW verdict against the champion's CV confidence interval.
5. Internalize the **singleness rule**: *two demo cases × one ceremony each = one ceremony per real-world business analysis*. Whenever you face a real business problem in the future, you open its test set ONCE — not multiple times.

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises** — one champion-pick exercise per case. Complete both before submitting your notebook.

---

## 💼 Why This Matters

Today is the payoff. Thirteen notebooks of locking discipline (test set sealed, all evaluation via cross-validation on the training set, every new model benchmarked against the Week-2 linear reference) converge into a single structured ceremony: **declare the protocol, compare candidates, pick a champion, open the locked envelope.**

Imagine you are presenting to **the State Health Department's review board** that has to approve MedScreen's breast-cancer screening pipeline, or to **HomeValue Analytics' deployment council** that has to sign off on the property-value prediction tool. Both rooms want the same three things from you:

1. A **defensible champion selection** — not "I tried a bunch of models and this one looked best", but *"every reference model introduced from nb01 onward was declared as a candidate (dummy floor, Week-2 linear, sparse linear, Ridge, single tree, random forest, default GBM, tuned GBM), evaluated under identical 5-fold CV folds with ROC-AUC (classification) or R² (regression) as the primary metric, and the champion was picked because its 95% confidence interval sat above the runner-up's by a clear margin — or, if the intervals overlapped, because it was the simpler model."*
2. A **95% confidence interval** on the champion's training-set performance — the headline number anyone in the room can quote.
3. A **single test-set evaluation** — the one and only authorized opening of the locked envelope — that confirms (or contradicts) what cross-validation predicted.

The third bullet is the part this notebook makes visible. Before today every cell that touched the test set was off-limits. **In §6 the test set opens — once, for each of the two business cases — and never again in the course.** The test set is for the final headline number you bring to the stakeholder meeting, not for iteration.

> **A question that often comes up here:** *"why is opening the test set such a big deal?"* Because every time you open the test set and let what you see influence what you do next, you have leaked the test set into model selection — which means the test set can no longer give you an unbiased estimate of how the model will perform on new data. The discipline is not statistical pedantry; it is the only mechanism by which the test-set point estimate has any meaning at all. nb14's ceremony exists to drive that point home in the most visible way possible: open the envelope, write the number down, close the envelope, never reopen.

---

## 1. Setup — Imports, References, Helpers

The setup cell does the same five jobs as nb12 / nb13 plus one new utility: `compare_models_comprehensive()`, the comparison harness that takes a dict of candidate models and returns a sorted DataFrame of CV means with Student's *t* 95% CIs.

> 💡 **Gemini Prompt:** "Set up the standard nb14 imports: pandas, numpy, matplotlib, scipy.stats, sklearn modules for `datasets` (`load_breast_cancer`, `fetch_california_housing`), `model_selection` (`train_test_split`, `cross_validate`, `StratifiedKFold`, `KFold`), `tree`, `ensemble`, `linear_model` (LogisticRegression, LinearRegression, Ridge, Lasso), `dummy` (DummyClassifier, DummyRegressor), `preprocessing.StandardScaler`, `pipeline.Pipeline`, and `metrics` (roc_auc_score, accuracy_score, f1_score, r2_score, mean_squared_error, mean_absolute_error). Suppress sklearn warnings, set `pd.set_option('display.precision', 4)`, `plt.rcParams['figure.figsize']=(10,6)`, and `RANDOM_SEED=474`. Define the course color constants `CLF_COLOR='#1f77b4'`, `REG_COLOR='#ff7f0e'`, `GREY='#999999'`, `GREEN='#2ca02c'`, `RED='#d62728'`. Build `reference_clf` and `reference_reg` as the Week-2 baseline pipelines (StandardScaler → `LogisticRegression(C=1.0)` for clf; StandardScaler → `LinearRegression()` for reg). Add the helpers carried over from previous notebooks: `plot_cv_ci` (dot plot with 95% Student's t CIs, df=4), `compare_models_comprehensive` (cross_validate wrapper that returns a DataFrame ranked by primary metric with per-metric mean/SD/folds plus fit time), and `verdict` (INSIDE/ABOVE/BELOW comparator against a CV CI)."
>
> **After running, verify:**
> - [ ] No import errors
> - [ ] `RANDOM_SEED` is set to 474
> - [ ] `reference_clf` and `reference_reg` are defined as `Pipeline` objects with `StandardScaler` + the Week-2 estimator
> - [ ] `plot_cv_ci`, `compare_models_comprehensive`, and `verdict` are all callable
> - [ ] Color constants `CLF_COLOR`, `REG_COLOR`, `GREY`, `GREEN`, `RED` are defined


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, KFold
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                              r2_score, mean_squared_error, mean_absolute_error)
import time, warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
CLF_COLOR = '#1f77b4'
REG_COLOR = '#ff7f0e'
GREY      = '#999999'
GREEN     = '#2ca02c'
RED       = '#d62728'

# --- Week-2 references ---
reference_clf = Pipeline([('scaler', StandardScaler()),
                          ('clf',    LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))])
reference_reg = Pipeline([('scaler', StandardScaler()),
                          ('reg',    LinearRegression())])

# --- Helper: CV-CI dot plot from previous notebooks ---
def plot_cv_ci(scores_dict, metric_name, title, ax, color=CLF_COLOR, k=5,
               highlight=None, highlight_color=GREEN):
    """Dot plot with 95% CIs; optionally highlight one model."""
    t_crit = stats.t.ppf(0.975, df=k - 1)
    rows = []
    for name, scores in scores_dict.items():
        m = float(np.mean(scores)); sd = float(np.std(scores, ddof=1))
        rows.append({'name': name, 'mean': m, 'half_w': t_crit * sd / np.sqrt(k)})
    df = pd.DataFrame(rows).sort_values('mean')
    colors = [highlight_color if (highlight is not None and r['name']==highlight) else color
              for _, r in df.iterrows()]
    for i, (_, r) in enumerate(df.iterrows()):
        ax.errorbar(r['mean'], i, xerr=r['half_w'], fmt='o', capsize=6, linewidth=2,
                    color=colors[i], markersize=10)
        ax.text(r['mean'] + r['half_w'] + (r['half_w']*0.2 if r['half_w']>0 else 0.001),
                i, f"{r['mean']:.4f}", va='center', fontsize=9)
    ax.set_yticks(range(len(df))); ax.set_yticklabels(df['name'])
    ax.set_xlabel(f'5-fold CV {metric_name} (mean ± 95% CI)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

# --- Comparison harness ---
def compare_models_comprehensive(models_dict, X, y, cv, scoring, primary_metric):
    """Cross-validate each model on the same folds; return DataFrame with means, SDs, fit times."""
    rows = []
    for name, model in models_dict.items():
        t0 = time.time()
        cvr = cross_validate(model, X, y, cv=cv, scoring=scoring,
                             return_train_score=False, n_jobs=-1)
        fit_time = time.time() - t0
        row = {'Model': name, 'fit_time_s': fit_time}
        for m in scoring:
            row[f'{m}_mean'] = cvr[f'test_{m}'].mean()
            row[f'{m}_sd']   = cvr[f'test_{m}'].std(ddof=1)
            row[f'{m}_folds'] = cvr[f'test_{m}']
        rows.append(row)
    df = pd.DataFrame(rows)
    return df.sort_values(f'{primary_metric}_mean', ascending=False).reset_index(drop=True)

# --- Verdict helper for the ceremony ---
def verdict(test_score, cv_mean, cv_sd, k=5):
    """Compare a test-set point estimate to a 95% CV CI; return INSIDE/ABOVE/BELOW + message."""
    t_crit = stats.t.ppf(0.975, df=k - 1)
    half_w = t_crit * cv_sd / np.sqrt(k)
    lo, hi = cv_mean - half_w, cv_mean + half_w
    if test_score < lo:
        return 'BELOW', f'Test ({test_score:.4f}) is BELOW the CV CI ({lo:.4f}, {hi:.4f}) — model overfit the training data.'
    if test_score > hi:
        return 'ABOVE', f'Test ({test_score:.4f}) is ABOVE the CV CI ({lo:.4f}, {hi:.4f}) — pleasant surprise; investigate why CV underestimated.'
    return 'INSIDE', f'Test ({test_score:.4f}) is INSIDE the CV CI ({lo:.4f}, {hi:.4f}) — CV estimate held; ship the model.'

print("✓ Setup, references, helpers, harness, verdict function — all loaded")


---

## 2. Load Both Datasets — Three Holdout Splits, Two Locked Envelopes

Same 60/20/20 splits with the same `random_state=RANDOM_SEED`. The CV scores you compute here will match (within fold-level fluctuation) the CV scores from previous notebooks because the splits are deterministic. The test envelopes have stayed sealed for three notebooks straight; today is the day they open.

> 💡 **Gemini Prompt:** "Apply the course's 60/20/20 split to both demo datasets. Load Wisconsin Breast Cancer with `load_breast_cancer(as_frame=True)` and California Housing with `fetch_california_housing(as_frame=True)`. For each: first carve off 20% as the LOCKED test set via `train_test_split(test_size=0.20, random_state=RANDOM_SEED)` — stratify on `y` for the classification case. Then split the remaining 80% into 75/25 to get train (60%) and val (20%). Build `cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)` and `cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)`. Print the train/val/test sizes per case with a note that the test envelopes stay locked until §6."
>
> **After running, verify:**
> - [ ] Classification split sizes are roughly 341 / 114 / 114 (train / val / test)
> - [ ] Regression split sizes are roughly 12384 / 4128 / 4128 (train / val / test)
> - [ ] `cv_clf` is a `StratifiedKFold(n_splits=5)` and `cv_reg` is a `KFold(n_splits=5)`
> - [ ] Both `X_test_*` envelopes are defined but unused below until §6


In [ ]:
data_clf = load_breast_cancer(as_frame=True)
X_clf, y_clf = data_clf.data, data_clf.target

# 60/20/20 split — identical partition to nb11 / nb12 / nb13 under RANDOM_SEED.
X_clf_temp, X_test_clf, y_clf_temp, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
X_train_clf, X_val_clf, y_train_clf, y_val_clf = train_test_split(
    X_clf_temp, y_clf_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_clf_temp
)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

data_reg = fetch_california_housing(as_frame=True)
X_reg, y_reg = data_reg.data, data_reg.target

X_reg_temp, X_test_reg, y_reg_temp, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
X_train_reg, X_val_reg, y_train_reg, y_val_reg = train_test_split(
    X_reg_temp, y_reg_temp, test_size=0.25, random_state=RANDOM_SEED
)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print(f'Classification: train {len(X_train_clf):>5} | val {len(X_val_clf):>4} | test {len(X_test_clf):>4} (LOCKED until §7.1)')
print(f'Regression:     train {len(X_train_reg):>5} | val {len(X_val_reg):>4} | test {len(X_test_reg):>4} (LOCKED until §7.2)')


---

## 3. The Model Selection Problem

Picking a model is harder than it sounds. The naïve approach — *"just compare the numbers and pick the highest one"* — has six classic failure modes. The right approach has six corresponding fixes:

| What goes wrong | What we do instead |
|---|---|
| Different CV splits per model (apples vs oranges) | **Same CV folds** for every candidate (apples vs apples) |
| Different metric per model ("metric shopping") | **One declared primary metric**, supporting metrics for context only |
| Pick the model by test-set score | Pick by cross-validation; **test set is for the final verdict, not selection** |
| Try new candidates until something wins | **Lock the candidate roster before fitting any model** |
| Ignore fit time | Track fit time as a tie-breaker when CV scores are close |
| No written rationale | Write a 3-sentence explanation in plain stakeholder language |

The protocol below operationalizes the right column. **Section 4** declares the candidate roster; **Section 5** evaluates everything on identical CV folds and visualizes the result; the **PAUSE-AND-DO exercises** ask you to pick a champion using the CI-overlap rule and explain it in stakeholder language; **Section 6** opens the locked test set.

> **A question that often comes up here:** *"isn't this overkill for a small problem? Can't I just compare scores and pick the highest?"* In a class assignment, sure — picking the highest score is fast, and the cost of being wrong is a grade. In a real business decision, *"I picked the highest score"* is the answer that loses you the room. The review board will ask which models you compared, on what data, with what metric, and how you decided the difference was real and not noise. A stakeholder who has lived through a model that looked great in development and embarrassed the company in production will not accept a one-sentence justification. The six-row table above is what a defensible answer looks like — and writing the protocol down *before* you fit any model is what keeps you honest later, when the numbers tempt you to skip steps.

---

## 4. Define the Candidate Roster — Every Reference Since nb01

The candidate roster is **declared before any results are seen**. This rules out "model shopping" — the failure mode where you keep trying new models until one happens to beat the rest. The discipline goes further today: **every reference model introduced from nb01 onward is on the roster, and each one carries the same hyperparameters its source notebook actually shipped**. nb14 is the chain's audit, not a re-tuning step.

Seven candidates per case for classification, eight for regression — the extra reg slot is the Ridge–Lasso pair from nb05.

| # | Family | Source notebook | Classification | Regression |
|---|---|---|---|---|
| 1 | **Dummy floor** (random-guess / mean) | nb03 (reg), nb06 (clf) | `DummyClassifier('most_frequent')` | `DummyRegressor('mean')` |
| 2 | **Week-2 reference** (linear) | nb06/nb08/nb09 (clf), nb03/nb08/nb09 (reg) | `LogReg(C=1.0)` | OLS (`LinearRegression`) |
| 3 | **Ridge regularization** (L2) | nb05 / nb08 | — *(L2 is already the Week-2 default for LogReg)* | `Ridge(alpha=1.0)` |
| 4 | **Sparse linear** (L1) | nb05 (reg), nb09 (clf) | `LogReg L1(C=0.1)` | `Lasso(alpha=0.01)` |
| 5 | **Single tree** | nb11 §6 | `DT(depth=3)` | `DT(depth=5)` |
| 6 | **Bagged ensemble** | nb12 §5 / §9 ship | `RF(n=50, max_features='sqrt', max_depth=3)` | `RF(n=50, max_features=0.5)` |
| 7 | **Boosted ensemble** (default) | nb13 §4 | `GBM(default)` | `GBM(default)` |
| 8 | **Boosted ensemble** (tuned) | nb13 §6 ship | `GBM(lr=0.2, n=200, max_features=0.5, max_depth=3)` | `GBM(lr=0.2, n=200, max_features=0.5, max_depth=3)` |

Three notes on the roster:

- The **dummy floor** is candidate #1 because every sophisticated candidate must clear it by a wide margin to justify its compute cost — `DummyClassifier('most_frequent')` is exactly the random-guess line (ROC-AUC = 0.500) and `DummyRegressor('mean')` is the predict-the-mean baseline (R² ≈ 0.000). Seeing the lift over the floor on the dot plot is a quick sanity check that the more elaborate models are buying real signal, not artifacts.
- The **Week-2 reference** is candidate #2 because it is the linear floor every later model has to clear by a *CI-clear* margin to earn displacement. The dummy is the *absolute* floor; the Week-2 reference is the *useful* floor.
- The classification roster has no separate "Ridge logistic" entry because `LogReg(C=1.0)` is already an L2-regularized linear classifier — the Week-2 reference IS Ridge logistic with the default C. The regression roster keeps Ridge and Lasso as separate entries because nb05 taught both as distinct regularization strategies.

**Hyperparameter provenance.** Rows 6, 7, and 8 use the **exact configurations nb12 and nb13 shipped**, not generic defaults:

- Row 6 — the bagged ensemble — uses nb12 §5's 3D joint-grid picks. **Classification**: `RandomForestClassifier(n=50, max_features='sqrt', max_depth=3)` — nb11's depth survives joint tuning under CI-overlap + parsimony; sklearn's clf default `max_features='sqrt'` paired with the smallest tree count. **Regression**: `RandomForestRegressor(n=50, max_features=0.5)` (max_depth defaults to None) — the §5 3D grid revealed depth-3 and depth-5 cells sit CI-clear below depth-None on California Housing; the depth-None tied group's largest-mean-smallest-CI cell at n=50 lives at `max_features=0.5`.
- Row 7 — the default GBM — uses sklearn's defaults exactly as nb13 §4 introduced them (`learning_rate=0.1, n_estimators=100, max_depth=3, max_features=None`).
- Row 8 — the tuned GBM — uses nb13 §6's joint 4D `(learning_rate × n_estimators × max_features × max_depth)` grid winners. **Both cases land on the same configuration**: `(lr=0.2, n_estimators=200, max_features=0.5, max_depth=3)` — sklearn's default GBM depth + nb12's regression `max_features` pick + the high-lr × many-trees corner of the §5 diagonal sweet spot.

> **A question that often comes up here:** *"do I have to memorize where every hyperparameter came from?"* No — the table is the **audit trail**, not a quiz. Its job is to let anyone in the room trace each candidate back to the source notebook where it was tuned, so the comparison is reproducible from scratch. When HomeValue's deployment council asks *"why is the GBM's `learning_rate` 0.2 and not 0.1?"* you point to nb13 §6's joint grid; when the State Health Department asks *"why is the random forest at 50 trees instead of 200?"* you point to nb12 §5. The names of the source notebooks and the headline hyperparameter values are what matters — the exact path through each joint grid is documented in the notebook itself and does not need to be in your head when you walk into the meeting.

> 💡 **Gemini Prompt:** "Declare two ordered dictionaries — `clf_models` and `reg_models` — that lock the candidate roster before any results are seen. Each value is the model object configured with the **exact hyperparameters its source notebook shipped**. **Classification (7 candidates):** `DummyClassifier('most_frequent')`, the Week-2 `reference_clf` pipeline, `LogReg L1` (`penalty='l1', C=0.1, solver='liblinear'`) wrapped in a StandardScaler pipeline, `DecisionTreeClassifier(max_depth=3)` [nb11 ship], `RandomForestClassifier(n_estimators=50, max_features='sqrt', max_depth=3, n_jobs=-1)` [nb12 §5 ship], `GradientBoostingClassifier()` [default, nb13 §4], and `GradientBoostingClassifier(n_estimators=200, learning_rate=0.2, max_features=0.5, max_depth=3)` [nb13 §6 ship]. **Regression (8 candidates):** `DummyRegressor('mean')`, the Week-2 `reference_reg` pipeline, `Ridge(alpha=1.0)` and `Lasso(alpha=0.01)` in StandardScaler pipelines, `DecisionTreeRegressor(max_depth=5)` [nb11 ship], `RandomForestRegressor(n_estimators=50, max_features=0.5, n_jobs=-1)` [nb12 §5 ship], `GradientBoostingRegressor()` [default, nb13 §4], and `GradientBoostingRegressor(n_estimators=200, learning_rate=0.2, max_features=0.5, max_depth=3)` [nb13 §6 ship]. Each candidate uses `random_state=RANDOM_SEED` where applicable. Print the candidate counts and confirm the roster is locked."
>
> **After running, verify:**
> - [ ] `clf_models` has exactly 7 entries; `reg_models` has exactly 8
> - [ ] Random Forest hyperparameters match nb12 §5's 3D joint-grid ship picks (clf: `(n=50, mf='sqrt', d=3)`; reg: `(n=50, mf=0.5)`)
> - [ ] Tuned GBM hyperparameters match nb13 §6's 4D joint-grid ship pick on both cases (`lr=0.2, n=200, mf=0.5, d=3`)
> - [ ] No additional candidates are added below this cell (roster is locked)


In [ ]:
clf_models = {
    'Dummy (most_frequent)':                          DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED),
    'Week-2 ref: LogReg(C=1.0)':                      reference_clf,
    'LogReg L1 (C=0.1)':                              Pipeline([('scaler', StandardScaler()),
                                                                ('clf', LogisticRegression(penalty='l1', C=0.1, solver='liblinear',
                                                                                           random_state=RANDOM_SEED, max_iter=5000))]),
    'Decision Tree (depth=3) [nb11 ship]':            DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED),
    'Random Forest (n=50, sqrt, d=3) [nb12 ship]':    RandomForestClassifier(n_estimators=50, max_features='sqrt', max_depth=3,
                                                                              random_state=RANDOM_SEED, n_jobs=-1),
    'GBM default [nb13]':                             GradientBoostingClassifier(random_state=RANDOM_SEED),
    'GBM tuned (lr=0.2, n=200, mf=0.5, d=3) [nb13 ship]': GradientBoostingClassifier(n_estimators=200, learning_rate=0.2,
                                                                                      max_features=0.5, max_depth=3,
                                                                                      random_state=RANDOM_SEED),
}

reg_models = {
    'Dummy (mean)':                                   DummyRegressor(strategy='mean'),
    'Week-2 ref: OLS':                                reference_reg,
    'Ridge (alpha=1.0)':                              Pipeline([('scaler', StandardScaler()),
                                                                ('reg', Ridge(alpha=1.0, random_state=RANDOM_SEED))]),
    'Lasso (alpha=0.01)':                             Pipeline([('scaler', StandardScaler()),
                                                                ('reg', Lasso(alpha=0.01, random_state=RANDOM_SEED, max_iter=5000))]),
    'Decision Tree (depth=5) [nb11 ship]':            DecisionTreeRegressor(max_depth=5, random_state=RANDOM_SEED),
    'Random Forest (n=50, mf=0.5) [nb12 ship]':       RandomForestRegressor(n_estimators=50, max_features=0.5,
                                                                             random_state=RANDOM_SEED, n_jobs=-1),
    'GBM default [nb13]':                             GradientBoostingRegressor(random_state=RANDOM_SEED),
    'GBM tuned (lr=0.2, n=200, mf=0.5, d=3) [nb13 ship]': GradientBoostingRegressor(n_estimators=200, learning_rate=0.2,
                                                                                     max_features=0.5, max_depth=3,
                                                                                     random_state=RANDOM_SEED),
}

print(f"✓ {len(clf_models)} classification candidates, {len(reg_models)} regression candidates declared")
print("✓ Random Forest and tuned GBM hyperparameters match the actual ship picks from nb12 §5 (3D joint grid) and nb13 §6 (4D joint grid)")
print("✓ Roster locked — no candidates added/removed after this cell")

---

## 5. Multi-Metric Reporting — CV Means with 95% CIs

For each case, evaluate every candidate under the **same** 5-fold CV folds. Track the primary metric plus two supporting metrics:

- **Classification:** ROC-AUC (primary, introduced in nb07), accuracy (nb06), F1 (nb07).
- **Regression:** R² (primary, introduced in nb03), RMSE (nb03), MAE (nb03).

Every metric is one the course material has already presented — nb14 does not invent new ones, and it does not introduce sklearn's `neg_*` scoring-string convention either. Instead, §5's code walks the CV folds manually with `cv_clf.split(X, y)` / `cv_reg.split(X, y)` and calls **nb03's `mean_absolute_error`, `mean_squared_error`, and `r2_score`** (plus **nb06's `accuracy_score`** and **nb07's `roc_auc_score` and `f1_score`**) directly on `(y_true, y_pred)` arrays for each fold. RMSE is computed post-hoc as `np.sqrt(mean_squared_error(...))`. That keeps the metric vocabulary one-to-one with the upstream notebooks.

The CV-CI dot plot for the primary metric is the **central visual** of the selection ceremony. The companion table reports all three metrics' means **and 95% Student's *t* CI half-widths** (df=4) per candidate, formatted as `mean ± half-width` with fixed-point notation (no scientific notation), plus the per-fit time. The runner-up question — *"is the top model's CI clearly above the runner-up's?"* — is what the next section adjudicates.

> **A question that often comes up here:** *"why three metrics per case instead of just the primary one?"* Because the primary metric drives the **verdict**, but the supporting metrics catch **blind spots** the primary one cannot see. On classification, ROC-AUC scores how well the model ranks positives above negatives — yet two models can have identical AUC and very different operating-point behavior; accuracy and F1 expose that. On regression, R² scores how much variance the model explains — yet two models with the same R² can have very different typical-error magnitudes; RMSE and MAE expose that. Reporting all three lets you say in the stakeholder meeting *"the champion beats the runner-up on the headline metric, and the supporting metrics agree — no hidden trade-off."* That sentence is much harder to dispute than a single number.


> 💡 **Gemini Prompt:** "Run a 5-fold CV comparison of every candidate in `clf_models` and `reg_models` *without* using sklearn's `neg_*` scoring strings. Walk the CV folds manually using `cv_clf.split(X_train_clf, y_train_clf)` and `cv_reg.split(X_train_reg, y_train_reg)`. For each (model, fold) pair, fit on the training portion of the fold and predict on the validation portion. **Classification:** call `roc_auc_score` on `predict_proba(...)[:,1]` (or fall back to NaN if the model lacks `predict_proba`), and call `accuracy_score`, `f1_score` on `predict(...)`. **Regression:** call `r2_score`, `mean_absolute_error`, and `np.sqrt(mean_squared_error(...))` directly — RMSE is computed post-hoc, not via a sklearn scoring string. Track each metric's per-fold array and fit time per fit. Compute mean + 95% Student's *t* CI half-width (df=4) per (candidate, metric) and stash them in `clf_results` / `reg_results` DataFrames with columns `Model`, `<metric>_mean`, `<metric>_sd`, `<metric>_ci`, `<metric>_folds`, and `fit_time_s` — sorted descending by the primary metric (ROC-AUC / R²). Build a separate display DataFrame per case where each metric column is formatted as `mean ± 95% CI half-width` to 4 decimal places (fixed-point, no scientific notation). Print both display tables. Then build `clf_dict` and `reg_dict` from the per-fold arrays and render the side-by-side CV-CI dot plots via `plot_cv_ci`."
>
> **After running, verify:**
> - [ ] The §5 code uses `mean_absolute_error`, `mean_squared_error`, `r2_score` (nb03) and `accuracy_score`, `roc_auc_score`, `f1_score` (nb06 / nb07) directly — no `neg_*` scoring strings
> - [ ] Both display tables show **mean ± 95% CI** per metric in fixed-point notation (no `1.234e-04` style scientific notation)
> - [ ] `clf_results` has 7 rows; `reg_results` has 8 rows
> - [ ] Both DataFrames are sorted descending by primary-metric mean
> - [ ] RMSE column reports positive values (square-root of MSE), not the `neg_` sklearn convention
> - [ ] Side-by-side dot plots render correctly


In [ ]:
# §5 — Multi-Metric Reporting using post-hoc metric computation.
# We walk the CV folds manually and call nb03/nb06/nb07's metric functions
# directly on (y_true, y_pred) arrays. No neg_* sklearn scoring strings —
# the metric vocabulary matches the upstream notebooks one-to-one.

def _evaluate_clf_per_fold(model, X, y, cv):
    """Per-fold ROC-AUC (nb07), accuracy (nb06), F1 (nb07) + fit time."""
    auc_folds, acc_folds, f1_folds, fit_times = [], [], [], []
    for tr_idx, va_idx in cv.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        t0 = time.time()
        model.fit(X_tr, y_tr)
        fit_times.append(time.time() - t0)
        if hasattr(model, 'predict_proba'):
            y_proba = model.predict_proba(X_va)[:, 1]
            auc_folds.append(roc_auc_score(y_va, y_proba))
        else:
            auc_folds.append(np.nan)
        y_pred = model.predict(X_va)
        acc_folds.append(accuracy_score(y_va, y_pred))
        f1_folds.append(f1_score(y_va, y_pred))
    return (np.array(auc_folds), np.array(acc_folds), np.array(f1_folds), float(np.mean(fit_times)))

def _evaluate_reg_per_fold(model, X, y, cv):
    """Per-fold R² (nb03), RMSE = sqrt(MSE) (nb03), MAE (nb03) + fit time."""
    r2_folds, rmse_folds, mae_folds, fit_times = [], [], [], []
    for tr_idx, va_idx in cv.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        t0 = time.time()
        model.fit(X_tr, y_tr)
        fit_times.append(time.time() - t0)
        y_pred = model.predict(X_va)
        r2_folds.append(r2_score(y_va, y_pred))
        rmse_folds.append(np.sqrt(mean_squared_error(y_va, y_pred)))  # post-hoc — nb03 form
        mae_folds.append(mean_absolute_error(y_va, y_pred))
    return (np.array(r2_folds), np.array(rmse_folds), np.array(mae_folds), float(np.mean(fit_times)))

def _ci(arr, k=5):
    """Mean and 95% Student's t CI half-width (df=k-1) for a CV fold array."""
    arr = np.asarray(arr)
    t_crit = stats.t.ppf(0.975, df=k - 1)
    return float(arr.mean()), float(arr.std(ddof=1)), float(t_crit * arr.std(ddof=1) / np.sqrt(k))

# === CLASSIFICATION ===
clf_rows = []
for name, model in clf_models.items():
    auc, acc, f1, ft = _evaluate_clf_per_fold(model, X_train_clf, y_train_clf, cv_clf)
    auc_m, auc_sd, auc_h = _ci(auc); acc_m, _, acc_h = _ci(acc); f1_m, _, f1_h = _ci(f1)
    clf_rows.append({
        'Model':         name,
        'roc_auc_folds': auc, 'roc_auc_mean': auc_m, 'roc_auc_sd': auc_sd, 'roc_auc_ci': auc_h,
        'accuracy_mean': acc_m, 'accuracy_ci': acc_h,
        'f1_mean':       f1_m, 'f1_ci':       f1_h,
        'fit_time_s':    ft,
    })
clf_results = pd.DataFrame(clf_rows).sort_values('roc_auc_mean', ascending=False).reset_index(drop=True)

# === REGRESSION ===
reg_rows = []
for name, model in reg_models.items():
    r2, rmse, mae, ft = _evaluate_reg_per_fold(model, X_train_reg, y_train_reg, cv_reg)
    r2_m, r2_sd, r2_h = _ci(r2); rmse_m, _, rmse_h = _ci(rmse); mae_m, _, mae_h = _ci(mae)
    reg_rows.append({
        'Model':      name,
        'r2_folds':   r2, 'r2_mean': r2_m, 'r2_sd': r2_sd, 'r2_ci': r2_h,
        'rmse_mean':  rmse_m, 'rmse_ci': rmse_h,
        'mae_mean':   mae_m, 'mae_ci': mae_h,
        'fit_time_s': ft,
    })
reg_results = pd.DataFrame(reg_rows).sort_values('r2_mean', ascending=False).reset_index(drop=True)

# === Display tables: mean ± 95% CI per metric, fixed-point notation ===
def _fmt(row, prefix, decimals=4):
    return f"{row[prefix + '_mean']:.{decimals}f} ± {row[prefix + '_ci']:.{decimals}f}"

clf_display = pd.DataFrame({
    'Model':                     clf_results['Model'],
    'ROC-AUC (mean ± 95% CI)':   clf_results.apply(lambda r: _fmt(r, 'roc_auc'),  axis=1),
    'Accuracy (mean ± 95% CI)':  clf_results.apply(lambda r: _fmt(r, 'accuracy'), axis=1),
    'F1 (mean ± 95% CI)':        clf_results.apply(lambda r: _fmt(r, 'f1'),       axis=1),
    'Fit time (s)':              clf_results['fit_time_s'].apply(lambda x: f"{x:.3f}"),
})
reg_display = pd.DataFrame({
    'Model':                     reg_results['Model'],
    'R² (mean ± 95% CI)':        reg_results.apply(lambda r: _fmt(r, 'r2'),   axis=1),
    'RMSE (mean ± 95% CI)':      reg_results.apply(lambda r: _fmt(r, 'rmse'), axis=1),
    'MAE (mean ± 95% CI)':       reg_results.apply(lambda r: _fmt(r, 'mae'),  axis=1),
    'Fit time (s)':              reg_results['fit_time_s'].apply(lambda x: f"{x:.3f}"),
})

print("=== CLASSIFICATION (5-fold CV, ranked by ROC-AUC) ===")
print(clf_display.to_string(index=False))
print()
print("=== REGRESSION (5-fold CV, ranked by R²) ===")
print(reg_display.to_string(index=False))

# Build per-candidate fold dicts for the CV-CI dot plots
clf_dict = {row['Model']: row['roc_auc_folds'] for _, row in clf_results.iterrows()}
reg_dict = {row['Model']: row['r2_folds']      for _, row in reg_results.iterrows()}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_cv_ci(clf_dict, 'ROC-AUC', 'Classification — 7 candidates ranked (full nb01-nb13 lineage)', axes[0], color=CLF_COLOR)
plot_cv_ci(reg_dict, 'R²',      'Regression — 8 candidates ranked (full nb01-nb13 lineage)',     axes[1], color=REG_COLOR)
fig.suptitle('Every reference model since nb01 on one plot, ranked by CV mean of the primary metric',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The dot plots are the central artifact of the selection ceremony. Each candidate's mean is the dot; the horizontal bar is its **95% confidence interval** (the range we'd expect to see in repeated cross-validation runs). The CI-overlap rule becomes a visual question — overlapping intervals mean statistical tie (simpler model wins); non-overlapping intervals mean the top model has earned displacement.

**The dummy floor anchors both panels.** `DummyClassifier('most_frequent')` lands at ROC-AUC = **0.500** — exactly the random-guess line — and `DummyRegressor('mean')` lands at R² ≈ **0.000** — the predict-the-mean baseline from nb03. Every sophisticated candidate clears the floor by a wide margin on both business cases, which is the quickest possible sanity check that the work is buying real signal rather than memorizing artifacts.

**The MedScreen verdict (classification).** Wisconsin breast cancer is a small dataset (\~341 training patients) and mostly linearly separable in standardized cell-nucleus space. The top candidates' confidence intervals overlap heavily: the Week-2 reference `LogReg(C=1.0)` leads at CV ROC-AUC ≈ **0.994** with a tight 95% CI \~**[0.986, 1.001]**; `LogReg L1` (\~0.990), `GBM tuned (lr=0.2, n=200, mf=0.5, d=3)` from nb13 §6 (\~0.986), the nb12 §5 ship pick `Random Forest (n=50, mf='sqrt', d=3)` (\~0.981), and default GBM (\~0.981) all sit inside that interval; only the depth-3 decision tree (CV ≈ 0.922) falls noticeably below. **By the CI-overlap parsimony rule, MedScreen's champion is the Week-2 `LogisticRegression(C=1.0)`** — the ensembles competed admirably but did not earn the right to displace the simplest linear model from nb06. This matches **both nb12 §9 and nb13 §7**'s verdict on the classification spine, which is exactly the kind of corroboration nb14's ceremony is designed to expose. For the State Health Department's review board, this is good news: the recommended model is the one with the most transparent decision boundary, which is what an oncologist can explain to a patient.

**The HomeValue verdict (regression).** California Housing is a 36× larger dataset and the picture is more spread out. The three linear candidates (OLS, Ridge α=1.0, Lasso α=0.01) cluster at CV R² ≈ **0.586** — Ridge and OLS are visually indistinguishable, empirically confirming nb09's grid-search finding that no Ridge α displaces OLS on this data. The single tree lifts to **0.613**; default GBM to **0.784**; and the top two candidates land in a close race: **tuned GBM `(lr=0.2, n=200, mf=0.5, d=3)` at CV R² ≈ 0.8117 with 95% CI [0.7983, 0.8251]**, and **Random Forest `(n=50, mf=0.5)` at CV R² ≈ 0.8038 with 95% CI [0.7937, 0.8138]**.

**Based on 5-fold cross-validation, the tuned GBM is selected as the final candidate model.** It achieved the highest mean R² of **0.8117** and the lowest mean RMSE of **0.4985**, while also producing the lowest MAE — although its MAE was nearly identical to the Random Forest model. Beyond the means, the tuned GBM's **CV CI lower bound (0.7983)** is *greater than* the Random Forest's CV CI lower bound (**0.7937**): even at its worst-case fold-to-fold draw, the tuned GBM outperforms the random forest's worst-case. That dominance on mean + worst-case CI is what breaks the upper-end overlap in the tuned GBM's favor.

Because the Random Forest model performed *very similarly*, **the GBM should be interpreted as a modest winner rather than an overwhelmingly superior model.** After selecting the tuned GBM, the model is retrained on the full development dataset (`X_train ∪ X_val`, 80% of the data) and then evaluated **once** on the locked test set to estimate final out-of-sample performance — that's §6.2's job.

> **A question that often comes up here:** *"if the random forest is so close, why ship the tuned GBM at all?"* Because a true statistical tie under the CI-overlap rule from nb08 requires both CIs to span comparable ranges — and here the tuned GBM's CV CI lower bound (0.7983) is **above** the random forest's (0.7937). The CIs do overlap on the upper end (RF's CI reaches 0.8138, GBM's reaches 0.8251), but the GBM's worst-case dominates the random forest's worst-case. Combined with the GBM's better mean R², better mean RMSE, and essentially-tied MAE, that gives the tuned GBM a small but defensible edge across every primary regression criterion the course teaches. The win is modest, not blowout — the random forest stays on the roster as a close-second runner-up, and HomeValue's deployment council should treat the gap as small enough to revisit if a future re-run flips the lower-CI ordering.

> **A question that often comes up here:** *"why bother including Ridge and the default GBM if neither becomes the champion?"* Because the runner-up rows are doing real teaching work. Ridge α=1.0 sitting on top of OLS confirms that nb09's empirical conclusion (no Ridge α dominates OLS on this data) generalizes from the random-search grid to today's ceremony. Default GBM landing well behind both the random forest and the tuned GBM quantifies what nb13's tuning bought — roughly 3 R² points and a CI-clear gap over the default. The full roster is a complete record of the course's modeling chain; deleting candidates because they are not the champion would erase the evidence behind the verdict.

**Key takeaway:** The expanded roster anchored to nb12 / nb13's actual ship picks produces two champions. **MedScreen ships `LogisticRegression(C=1.0)`** — the ensembles tie the Week-2 reference under CI-overlap, and parsimony picks the simpler linear model. **HomeValue ships the tuned `GradientBoostingRegressor(learning_rate=0.2, n_estimators=200, max_features=0.5, max_depth=3)`** — the modest winner on mean R², mean RMSE, mean MAE (≈ tied), and CV CI lower bound; the random forest is the close-second runner-up. The PAUSE-AND-DO exercises below ask you to apply the CI-overlap rule with the dominance tiebreaker yourself, set the champion variables for the §6 ceremony, and write a 3-sentence stakeholder explanation for each case.

---

## 📝 PAUSE-AND-DO Exercise 1 (Classification, 7 minutes) — Pick the Champion and Explain Why

The §5 dot plot just showed you seven candidates ranked by CV ROC-AUC. Your job: apply the **CI-overlap rule** to pick the classification champion, set the variables §6's ceremony will need, then explain the pick in stakeholder language for the State Health Department's review board.

**Steps:**

1. From `clf_results`, pull the **top row by mean** (the champion candidate) and the **runner-up** (second row).
2. Compute the **95% CI half-width** for both: `t_crit × sd / √k` where `t_crit = scipy.stats.t.ppf(0.975, df=4)` and `k = 5`.
3. Check whether the two 95% CIs overlap.
   - **CIs overlap** → statistical tie → the **simpler model wins** by parsimony (use `fit_time_s` as the tie-breaker when both are linear-ish).
   - **CIs do NOT overlap** → the top model wins outright (CI-clear margin).
4. Set `champ_clf_name`, `champ_clf_mean`, `champ_clf_sd` to the chosen champion's `Model`, `roc_auc_mean`, and `roc_auc_sd`. **The §6 ceremony cell will not run without these variables.**
5. Below the code cell, fill in the markdown template with a **3-sentence explanation** the State Health Department's review board could read aloud.

---

> 💡 **Gemini Prompt:** "From `clf_results` (a pandas DataFrame already ranked by `roc_auc_mean` in descending order), pull `iloc[0]` (champion candidate) and `iloc[1]` (runner-up). Compute the 95% CI half-width for each using `t_crit = scipy.stats.t.ppf(0.975, df=4)` and `k = 5`. Check whether the two intervals overlap. If they do, pick the simpler model (use the row with lower `fit_time_s` as the tie-breaker) — if not, keep the top row as champion. Set `champ_clf_name`, `champ_clf_mean`, `champ_clf_sd` accordingly. Print the champion's name, its 95% CI bounds, and a one-line verdict explaining whether the rule kept the top row or swapped to the runner-up."
>
> **After running, verify:**
> - [ ] `champ_clf_name`, `champ_clf_mean`, `champ_clf_sd` are all defined
> - [ ] One-line CI-overlap verdict is printed
> - [ ] Champion's 95% CI is printed (e.g., `[0.986, 1.001]`)

In [ ]:
# YOUR SOLUTION CODE HERE
# Identify the classification champion using the CI-overlap rule.
# Required output variables (used by §6 ceremony):
#   champ_clf_name : str
#   champ_clf_mean : float
#   champ_clf_sd   : float


## 📝 PAUSE-AND-DO Exercise 2 (Regression, 7 minutes) — Pick the Champion and Explain Why

Same template, the regression case. The §5 dot plot showed you eight candidates ranked by CV R². Apply the **CI-overlap rule** to pick the regression champion, set the variables §6's ceremony will need, then explain the pick for HomeValue Analytics' deployment council.

**Steps:**

1. From `reg_results`, pull the **top row** (champion candidate) and the **runner-up** (second row).
2. Compute the 95% CI half-width for both: `t_crit × sd / √k` where `t_crit = scipy.stats.t.ppf(0.975, df=4)` and `k = 5`.
3. Apply the **CV selection rule** (CI-overlap + dominance tiebreaker).
   - **CIs do NOT overlap** → top model wins outright (CI-clear margin).
   - **CIs overlap AND top's CV CI lower bound > runner-up's** → **top is the modest winner** (top dominates on both mean and worst-case CI bound; ship the top).
   - **CIs overlap AND top's CV CI lower bound ≤ runner-up's** → statistical tie under strict CI-overlap → simpler model wins by parsimony (lower `fit_time_s` as tie-breaker).
4. Set `champ_reg_name`, `champ_reg_mean`, `champ_reg_sd`. **The §6 ceremony cell will not run without these variables.**
5. Below the code cell, fill in the markdown template. Convert R² to a USD-RMSE intuition when justifying — *"the champion's CV R² is X, which translates to roughly USD Y of typical prediction error per property"*. Tie the revisit trigger to housing-market dynamics (median income shifts, new construction permits, etc.).

---

> 💡 **Gemini Prompt:** "From `reg_results` (a pandas DataFrame already ranked by `r2_mean` in descending order), pull `iloc[0]` (top candidate) and `iloc[1]` (runner-up). Compute the 95% CI half-width and the CI lower bound for each using `t_crit = scipy.stats.t.ppf(0.975, df=4)` and `k = 5`. Apply the modest-winner rule: if the top model's CI lower bound exceeds the runner-up's CI lower bound, top dominates on both mean and worst-case → pick the top as the modest winner. Otherwise, fall back to CI-overlap parsimony (pick whichever has lower `fit_time_s`). Set `champ_reg_name`, `champ_reg_mean`, `champ_reg_sd`. Print the champion name, its 95% CI bounds, the equivalent CV-RMSE in USD (multiply the RMSE column by 100,000), and a one-line verdict naming which branch of the rule fired."
>
> **After running, verify:**
> - [ ] `champ_reg_name`, `champ_reg_mean`, `champ_reg_sd` are all defined
> - [ ] On California Housing the modest-winner rule picks the **tuned GBM** (top.lower_CI 0.7983 > runner.lower_CI 0.7937)
> - [ ] Champion's 95% CI is printed in R² units AND CV-RMSE is printed in USD

In [ ]:
# YOUR SOLUTION CODE HERE
# Identify the regression champion using the CI-overlap rule.
# Required output variables (used by §6 ceremony):
#   champ_reg_name : str
#   champ_reg_mean : float
#   champ_reg_sd   : float


## 6. Finalize the Champion and Open the Locked Test Set

§5 picked the champion. §6 answers the deployment question every stakeholder asks: **"How well should I expect this finalized model to perform on genuinely unseen data?"** Two steps, in order — and only those two.

### Step 1 — Finalize the champion on every drop of development data (train + val)

The champion's model class and hyperparameters were chosen in §5 via 5-fold cross-validation on `X_train_*` (60% of the data). The `X_val_*` envelope (20%) was set aside at the start of nb14 but has **not been touched** during selection — CV did its work on the training pool alone. With the champion now committed, validation has no further job, so we **refit the champion on the combined development pool `X_dev = X_train ∪ X_val`** (80% of the data) before we touch the test envelope.

Why retrain on train+val?

1. **More data → better deployment.** The shipped model should learn from every row we have made available to it. Holding back 20% of the data on a finalized model adds no value — selection is finished and validation has no further role.
2. **Honest pipeline.** The standard ML pattern is *"develop on train → validate hyperparameter choices via CV → finalize on train+val → evaluate on test."* Anything else either under-trains the shipped model or leaks the test envelope.

A small but predictable consequence: because the deployed model trains on more data than CV did, the test-set score will often land *slightly above* the CV CI. That is the conservative-CV story working as intended, not a bug — read it as "CV was modestly underestimating the deployed model's performance because CV trained on 60% of the data while the deployment trains on 80%."

### Step 2 — Open the locked test set ONCE, score the finalized champion, render the verdict

The test envelope opens here for the first and only time in this course. Predict on `X_test_*`, compute the test-set point estimate of the primary metric, compare against the §5 CV 95% CI, and pronounce the categorical verdict:

- **INSIDE** → the test point lands within the CV CI. Cross-validation was honest. Ship the model.
- **ABOVE** → the test point lands above the CV CI's upper bound. A small drift is expected (Step 1's train+val refit lifts deployment performance a touch); a large drift needs investigation before deploying.
- **BELOW** → the test point lands below the CV CI's lower bound. **Do not deploy.** Diagnose the development pipeline (most often a leakage problem) and re-open the ceremony with a fresh test set.

§6.3 below carries the full playbook for each verdict — what to record, what to investigate, what to deploy or block, and what the *test-set-is-spent* rule says about retrying.

For HomeValue Analytics specifically, the champion entering §6.2 is the **tuned gradient-boosting machine** `GradientBoostingRegressor(learning_rate=0.2, n_estimators=200, max_features=0.5, max_depth=3)` — the §5 modest winner. It beats nb12's random forest on mean R², mean RMSE, mean MAE (essentially tied), and CV CI lower bound; the random forest stays on the roster as the close-second runner-up.

### Why NOT score every candidate on the test set and pick the best?

It is tempting to skip §5's CV ranking and just run every candidate through the test set:

```
Candidate Model 1 → test performance
Candidate Model 2 → test performance
Candidate Model 3 → test performance
Candidate Model 4 → test performance
Candidate Model 5 → test performance
Choose the best test-set result.
```

**Do not do this.** Picking the best of N test scores **turns the test set into another validation set** — the chosen model was indirectly selected *because it looked good on the test sample*. The winner's reported test score then becomes **optimistically biased**: on a fresh dataset the score regresses back toward the mean of the candidates, often substantially. The whole point of the locked test set is to give an **unbiased** estimate of deployment performance; the moment any candidate's test score influences which model ships, the test set has done two jobs (selection + evaluation), and **neither estimate is honest anymore**.

The discipline is binary: **§5's CV picks the champion**; **§6 scores that committed champion ONCE on test**. The two activities never mix.

> **A question that often comes up here:** *"why bother with Step 1's train+val refit — can't I just evaluate the §5-tuned champion on the test set directly?"* Because the champion you tuned in §5 only saw 60% of the data during cross-validation. The model the stakeholder will deploy should learn from every row of development data available — that is the 80% pool, train + val together. Skipping the refit means the test-set number you report is for a model that is *not* the one that ships, which violates the basic honest-pipeline rule. The cost of the refit is one extra `model.fit(...)` call before the test prediction; the cost of skipping it is a headline number that does not describe what the production model will actually do once deployed.

### 6.1 Classification Ceremony — Wisconsin Breast Cancer

The MedScreen classification ceremony evaluates the committed champion against `X_test_clf` — the 114 patients held out since §2. **Step 1** refits the champion on `X_dev_clf = X_train_clf ∪ X_val_clf` (455 patients = 341 train + 114 val). **Step 2** opens the locked envelope for the one and only authorized prediction pass, computes test ROC-AUC (primary), accuracy, and F1, and renders the INSIDE / ABOVE / BELOW verdict against the §5 CV CI.

Under §5's CI-overlap rule the champion is whichever model PAUSE-AND-DO 1 named: the top-by-mean candidate is `LogReg(C=1.0)` at CV ROC-AUC ≈ 0.994, but its CI overlaps the runners-up — `LogReg L1`, the tuned GBM, the Random Forest, and the default GBM — so parsimony picks the simplest model class (the Week-2 linear baseline). If Exercise 1 was skipped, §6.1's auto-pick fallback applies the same rule and lands on a CI-tied linear variant via the `fit_time_s` secondary tiebreaker.

Two values matter when reading the output below: the **CV mean ± 95% CI** (a forecast of deployment performance) and the **test-set point** (a single honest observation against that forecast). The money plot puts them side by side so the verdict is visual, not arithmetic.

> **A question that often comes up here:** *"who at MedScreen actually sees the test-set number, and what do they do with it?"* The State Health Department's review board sees it as the headline metric in the approval package — *"on a held-out set of 114 patients the screening model achieved test ROC-AUC of X, inside the CV 95% CI of [Y, Z]."* The board does not run the model itself; it reads the number, compares it to the regulatory threshold and the prior screening protocol's baseline, and decides whether to authorize the rollout. The INSIDE / ABOVE / BELOW verdict tells the board whether the CV story held up — if INSIDE, the rollout proceeds on the CV evidence; if BELOW, the package goes back for diagnosis before any patient sees the model.

> 💡 **Gemini Prompt:** "Open the LOCKED classification test set EXACTLY ONCE. First, fall back to an auto-pick if Exercise 1 left `champ_clf_name` / `champ_clf_mean` / `champ_clf_sd` undefined: apply the CI-overlap rule (top row vs runner-up of `clf_results`; if 95% CIs overlap and runner-up has lower `fit_time_s`, simpler wins — else top stays), set the champion variables, and print a `⚠️  Exercise 1 left empty — auto-picked ...` warning. **Step 1 — finalize**: build `X_dev_clf = pd.concat([X_train_clf, X_val_clf])` and `y_dev_clf = pd.concat([y_train_clf, y_val_clf])` (the combined development pool — 80% of the data) and refit `champion_clf = clf_models[champ_clf_name]` on `(X_dev_clf, y_dev_clf)`. **Step 2 — score on test**: predict probabilities and classes on `X_test_clf`, compute test ROC-AUC / accuracy / F1. Call `verdict(test_auc, champ_clf_mean, champ_clf_sd)` to get an INSIDE / ABOVE / BELOW pronouncement. Print a header block with champion name, CV mean ± SD, dev-fit size (rows), test scores, and the verdict. Then render the **money plot**: a dot at `champ_clf_mean` with horizontal error bar `±t_crit·sd/√5` (df=4); a triangle at `test_auc` colored by verdict (GREEN=INSIDE, CLF_COLOR=ABOVE, RED=BELOW); dotted vertical lines at the CI bounds. Title: 'Classification ceremony — test ROC-AUC vs CV CI of the champion ({champ_clf_name})'."
>
> **After running, verify:**
> - [ ] Step 1 fits the champion on the combined development pool (train + val), not on train alone — `X_dev_clf` has \~80% of rows
> - [ ] If exercises were skipped: the auto-pick warning prints with the chosen champion name
> - [ ] Header block prints champion name, CV mean ± SD, dev-fit row count, test ROC-AUC / accuracy / F1, and the INSIDE / ABOVE / BELOW verdict
> - [ ] Money plot shows the CV-CI error bar and the test-point triangle, colored by verdict


In [ ]:
# CEREMONY — CLASSIFICATION
# X_test_clf opens here, exactly once. After this cell, it goes back into the envelope.

# Fallback: if Exercise 1 left the champion variables undefined (YOUR SOLUTION
# CODE HERE was not filled in), auto-pick the champion via the CI-overlap rule
# so the ceremony still runs end-to-end.
if 'champ_clf_name' not in globals() or 'champ_clf_mean' not in globals() or 'champ_clf_sd' not in globals():
    _top    = clf_results.iloc[0]
    _runner = clf_results.iloc[1]
    _t      = stats.t.ppf(0.975, df=4)
    _h_top  = _t * _top['roc_auc_sd']    / np.sqrt(5)
    _h_run  = _t * _runner['roc_auc_sd'] / np.sqrt(5)
    _overlap = (max(_top['roc_auc_mean'] - _h_top, _runner['roc_auc_mean'] - _h_run)
                <= min(_top['roc_auc_mean'] + _h_top, _runner['roc_auc_mean'] + _h_run))
    _pick = _runner if (_overlap and _runner['fit_time_s'] < _top['fit_time_s']) else _top
    champ_clf_name = _pick['Model']
    champ_clf_mean = _pick['roc_auc_mean']
    champ_clf_sd   = _pick['roc_auc_sd']
    print(f"⚠️  Exercise 1 left empty — auto-picked champion via CI-overlap rule: {champ_clf_name}")
    print()

champion_clf = clf_models[champ_clf_name]

# Step 1 — FINALIZE: refit the champion on the combined development pool
# (train + val). Selection is finished; val has no further role. The deployed
# model should learn from every row of development data we have available.
X_dev_clf = pd.concat([X_train_clf, X_val_clf])
y_dev_clf = pd.concat([y_train_clf, y_val_clf])
champion_clf.fit(X_dev_clf, y_dev_clf)

# Step 2 — SCORE on the locked test set, exactly ONCE.
y_proba_test_clf = champion_clf.predict_proba(X_test_clf)[:, 1]
y_pred_test_clf  = champion_clf.predict(X_test_clf)

test_auc      = roc_auc_score(y_test_clf, y_proba_test_clf)
test_accuracy = accuracy_score(y_test_clf, y_pred_test_clf)
test_f1       = f1_score(y_test_clf, y_pred_test_clf)

verdict_label_clf, verdict_msg_clf = verdict(test_auc, champ_clf_mean, champ_clf_sd)

print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'CEREMONY — CLASSIFICATION (Wisconsin Breast Cancer)')
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'Champion:            {champ_clf_name}')
print(f'CV ROC-AUC mean:     {champ_clf_mean:.4f}  (SD = {champ_clf_sd:.4f})')
print(f'Step 1 dev-fit rows: {len(X_dev_clf)}  (= train {len(X_train_clf)} + val {len(X_val_clf)})')
print(f'Test ROC-AUC:        {test_auc:.4f}')
print(f'Test accuracy:       {test_accuracy:.4f}')
print(f'Test F1:             {test_f1:.4f}')
print(f'')
print(f'VERDICT: {verdict_label_clf}')
print(f'{verdict_msg_clf}')

# THE money plot for classification
t_crit = stats.t.ppf(0.975, df=4)
half_w = t_crit * champ_clf_sd / np.sqrt(5)
fig, ax = plt.subplots(figsize=(11, 4))
verdict_color = {'INSIDE': GREEN, 'ABOVE': CLF_COLOR, 'BELOW': RED}[verdict_label_clf]
ax.errorbar([champ_clf_mean], [0], xerr=[half_w], fmt='o', capsize=10,
            color=GREY, markersize=12, linewidth=3, label=f'CV mean ± 95% CI: {champ_clf_mean:.4f} ± {half_w:.4f}')
ax.scatter([test_auc], [0], color=verdict_color, marker='^', s=300, zorder=5,
           label=f'Test point: {test_auc:.4f}  →  {verdict_label_clf}')
ax.axvline(champ_clf_mean - half_w, color=GREY, linestyle=':', alpha=0.5)
ax.axvline(champ_clf_mean + half_w, color=GREY, linestyle=':', alpha=0.5)
ax.set_yticks([]); ax.set_xlabel('ROC-AUC')
ax.set_title(f'Classification ceremony — test ROC-AUC vs CV CI of the champion ({champ_clf_name})',
             fontsize=12, fontweight='bold')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


**Reading the output:**

The triangle is the test-set point estimate; the dot is the CV mean; the dotted vertical lines mark the 95% CI bounds. If the triangle is between the dotted lines (INSIDE), the CV estimate held — ship the model. If the triangle is above the upper dotted line (ABOVE), the test set was kinder to the model than CV expected; on this notebook a small ABOVE drift is **expected** because Step 1 refit the champion on train+val (80% of the data) while CV only saw the train slice (60%) — investigate but do not panic. If the triangle is below the lower dotted line (BELOW), the model overfit the training data and the test-set number is the honest one.

Most of the time the verdict is INSIDE because CV-with-CI is well-calibrated under i.i.d. assumptions. ABOVE is the typical outcome of the train+val refit and small datasets. BELOW is the dangerous case — it usually signals a leakage problem in the training pipeline that CV did not catch.

**The classification test set is now closed. It will not reopen in this course.**

> **A question that often comes up here:** *"what do I say to MedScreen's review board if the verdict is ABOVE the CV upper bound?"* You explain *why a small ABOVE drift is expected here*: the shipped model trained on 455 patients (train + val) while CV trained on \~341 per fold, so the deployed model saw about 33% more data than CV measured. A small ABOVE drift is the conservative-CV story working as intended — CV was modestly underestimating the deployed model's performance because it trained on less data. If the ABOVE gap were large (say, more than two CI widths), you would investigate — but a hair above the upper bound is a positive signal, not a red flag, and the board will accept that explanation as long as you have it written down before the meeting starts.

---


### 6.2 Regression Ceremony — California Housing

Same two-step protocol on the regression side. **Step 1**: refit the champion on the combined development pool `X_dev_reg = X_train_reg ∪ X_val_reg` (16,512 rows = 12,384 train + 4,128 val). **Step 2**: open the locked test set `X_test_reg` for exactly one prediction pass and render the INSIDE / ABOVE / BELOW verdict against the §5 CV CI.

**The regression champion is the tuned gradient-boosting machine** `GradientBoostingRegressor(learning_rate=0.2, n_estimators=200, max_features=0.5, max_depth=3)` — nb13 §6's pick. **Based on 5-fold cross-validation, the tuned GBM was selected as the final candidate model.** It achieved the highest mean R² of 0.8117 and the lowest mean RMSE of 0.4985, while also producing the lowest MAE (essentially tied with the Random Forest at the third decimal). Its CV CI lower bound (0.7983) is also above the Random Forest's CV CI lower bound (0.7937) — even the GBM's worst-case beats the RF's worst-case. Because the Random Forest model performed *very similarly*, **the GBM should be interpreted as a modest winner rather than an overwhelmingly superior model.** §6.2 retrains the tuned GBM on the full development set and evaluates it once on the locked test set to estimate final out-of-sample performance.

> **A question that often comes up here:** *"if the tuned GBM only beats the random forest by a modest margin, why not just ship the simpler random forest?"* Because the dominance tiebreaker is not a tie at all — the tuned GBM dominates the random forest on **mean and worst-case CI bound**, not just one of the two. That is the threshold the course teaches for *"the top model has earned the win, even if the win is not a CI-clear blowout."* HomeValue's deployment council should hear this as *"the GBM is the data-driven pick, but the random forest is a credible fallback if the GBM ever shows trouble in production."* Keeping the runner-up on the dashboard as a backup model is a sensible operational practice; declaring a tie and shipping the simpler model when the data shows a consistent edge for the more complex one is leaving accuracy on the table.


> 💡 **Gemini Prompt:** "Open the LOCKED regression test set EXACTLY ONCE. **Apply the §5 modest-winner rule deterministically** before fitting: pull `_top = reg_results.iloc[0]` (highest mean) and `_runner = reg_results.iloc[1]`; compute their 95% CI half-widths via `t_crit = scipy.stats.t.ppf(0.975, df=4)`; pick `_top` if its CI lower bound exceeds `_runner`'s CI lower bound (top dominates on both mean AND worst-case → modest winner); otherwise fall back to CI-overlap parsimony (simpler model wins by `fit_time_s` tiebreaker). On California Housing the rule picks the tuned GBM — `_top` is the tuned GBM with mean 0.8117 and CI lower 0.7983, vs the Random Forest at mean 0.8038 and CI lower 0.7937. Set `champ_reg_name` / `champ_reg_mean` / `champ_reg_sd` from the rule's pick (the tuned GBM). If Exercise 2 had named a different model, print an `ℹ️  Exercise 2 picked X but the §5 modest-winner rule picks Y; ceremony applies Y` note and continue. **Step 1 — finalize**: build `X_dev_reg = pd.concat([X_train_reg, X_val_reg])` and `y_dev_reg = pd.concat([y_train_reg, y_val_reg])` (16,512 rows = 12,384 train + 4,128 val) and refit `champion_reg = reg_models[champ_reg_name]` on `(X_dev_reg, y_dev_reg)`. **Step 2 — score on test**: predict on `X_test_reg`, compute test R² / RMSE / MAE. Multiply RMSE and MAE by 100,000 to translate California Housing's target units (USD 100K) into dollars. Call `verdict(test_r2, champ_reg_mean, champ_reg_sd)` for the INSIDE / ABOVE / BELOW pronouncement. Print a header block with champion name, CV R² mean ± SD, dev-fit row count, test R² / RMSE / MAE (raw + USD), and the verdict. Render the **regression money plot**: dot at `champ_reg_mean` with horizontal error bar `±t_crit·sd/√5` (df=4); triangle at `test_r2` colored by verdict (GREEN=INSIDE, REG_COLOR=ABOVE, RED=BELOW); dotted vertical lines at the CI bounds. Title: 'Regression ceremony — test R² vs CV CI of the champion ({champ_reg_name})'."
>
> **After running, verify:**
> - [ ] The §5 modest-winner rule is what drives the choice: top.lower_CI (0.7983) > runner.lower_CI (0.7937), so top wins outright
> - [ ] If Exercise 2's pick differed from the rule, an `ℹ️  Exercise 2 picked X but the §5 rule picks Y` note prints
> - [ ] Step 1 fits the champion on `X_dev_reg` (train + val, 16,512 rows), not on train alone
> - [ ] `X_test_reg` opens only in this cell
> - [ ] Header block prints champion name (tuned GBM), CV R² mean ± SD, dev-fit row count, test R² / RMSE / MAE (raw + USD), and the INSIDE / ABOVE / BELOW verdict


In [ ]:
# CEREMONY — REGRESSION
# X_test_reg opens here, exactly once. After this cell, it goes back into the envelope.
#
# §6.2 ALWAYS derives the champion via the §5 modest-winner rule directly,
# rather than relying on whatever Exercise 2 set. The rule is:
#   top-by-mean wins outright if its CV CI lower bound > runner-up's CV CI
#   lower bound (top dominates on mean AND worst-case);
#   otherwise CI-overlap parsimony applies (simpler model wins by lower
#   fit_time_s).
# On California Housing the tuned GBM dominates the random forest on both
# mean and worst-case CI bound (0.8117 vs 0.8038 mean; 0.7983 vs 0.7937
# lower CI), so the rule picks the tuned GBM — modest winner, not blowout.

_top    = reg_results.iloc[0]   # highest mean
_runner = reg_results.iloc[1]   # runner-up
_t      = stats.t.ppf(0.975, df=4)
_h_top  = _t * _top['r2_sd']    / np.sqrt(5)
_h_run  = _t * _runner['r2_sd'] / np.sqrt(5)
top_lower    = _top['r2_mean']    - _h_top
runner_lower = _runner['r2_mean'] - _h_run

if top_lower > runner_lower:
    # Top dominates: higher mean AND higher worst-case CI bound → modest winner.
    _rule_pick = _top
else:
    # Runner-up has narrower CI / better worst-case → CI-overlap parsimony.
    _rule_pick = _runner if _runner['fit_time_s'] < _top['fit_time_s'] else _top

# If Exercise 2 set a different champion, note the mismatch and use the rule.
if 'champ_reg_name' in globals() and champ_reg_name != _rule_pick['Model']:
    print(f"ℹ️  Exercise 2 picked '{champ_reg_name}' but the §5 modest-winner rule selects "
          f"'{_rule_pick['Model']}'.")
    print(f"   Rule: top.lower_CI ({top_lower:.4f}) {'>' if top_lower > runner_lower else '<='} runner.lower_CI ({runner_lower:.4f}).")
    print(f"   The ceremony applies the rule's pick so the test set always evaluates the documented ship pick.")
    print()

champ_reg_name = _rule_pick['Model']
champ_reg_mean = _rule_pick['r2_mean']
champ_reg_sd   = _rule_pick['r2_sd']

champion_reg = reg_models[champ_reg_name]

# Step 1 — FINALIZE: refit the champion on the combined development pool
# (train + val). Selection is finished; val has no further role. The deployed
# model should learn from every row of development data we have available.
X_dev_reg = pd.concat([X_train_reg, X_val_reg])
y_dev_reg = pd.concat([y_train_reg, y_val_reg])
champion_reg.fit(X_dev_reg, y_dev_reg)

# Step 2 — SCORE on the locked test set, exactly ONCE.
y_pred_test_reg = champion_reg.predict(X_test_reg)

test_r2   = r2_score(y_test_reg, y_pred_test_reg)
test_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_test_reg))
test_mae  = mean_absolute_error(y_test_reg, y_pred_test_reg)

verdict_label_reg, verdict_msg_reg = verdict(test_r2, champ_reg_mean, champ_reg_sd)

print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'CEREMONY — REGRESSION (California Housing)')
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'Champion:            {champ_reg_name}')
print(f'CV R² mean:          {champ_reg_mean:.4f}  (SD = {champ_reg_sd:.4f})')
print(f'Step 1 dev-fit rows: {len(X_dev_reg)}  (= train {len(X_train_reg)} + val {len(X_val_reg)})')
print(f'Test R²:             {test_r2:.4f}')
print(f'Test RMSE:           {test_rmse:.4f}  (USD {test_rmse * 100_000:,.0f})')
print(f'Test MAE:            {test_mae:.4f}   (USD {test_mae  * 100_000:,.0f})')
print(f'')
print(f'VERDICT: {verdict_label_reg}')
print(f'{verdict_msg_reg}')

# Money plot for regression
t_crit = stats.t.ppf(0.975, df=4)
half_w_r = t_crit * champ_reg_sd / np.sqrt(5)
fig, ax = plt.subplots(figsize=(11, 4))
verdict_color = {'INSIDE': GREEN, 'ABOVE': REG_COLOR, 'BELOW': RED}[verdict_label_reg]
ax.errorbar([champ_reg_mean], [0], xerr=[half_w_r], fmt='o', capsize=10,
            color=GREY, markersize=12, linewidth=3,
            label=f'CV mean ± 95% CI: {champ_reg_mean:.4f} ± {half_w_r:.4f}')
ax.scatter([test_r2], [0], color=verdict_color, marker='^', s=300, zorder=5,
           label=f'Test point: {test_r2:.4f}  →  {verdict_label_reg}')
ax.axvline(champ_reg_mean - half_w_r, color=GREY, linestyle=':', alpha=0.5)
ax.axvline(champ_reg_mean + half_w_r, color=GREY, linestyle=':', alpha=0.5)
ax.set_yticks([]); ax.set_xlabel('R²')
ax.set_title(f'Regression ceremony — test R² vs CV CI of the champion ({champ_reg_name})',
             fontsize=12, fontweight='bold')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


**Reading the output:**

The header block names the champion the ceremony evaluated — the **tuned gradient-boosting machine** `GradientBoostingRegressor(learning_rate=0.2, n_estimators=200, max_features=0.5, max_depth=3)` from nb13 §6. That is **always** the regression ship pick on this notebook, regardless of what Exercise 2 named, because §6.2 derives the champion via §5's modest-winner rule directly: the tuned GBM has higher mean R² *and* a higher CV CI lower bound than the random forest, so it dominates on both expected value and worst-case CI bound. If Exercise 2 set a different pick (a strict CI-overlap parsimony reading would land on the Random Forest), the ceremony prints an `ℹ️  Exercise 2 picked ... but the §5 rule picks the tuned GBM; the ceremony applies the rule's pick` note — that lets you see exactly where your exercise interpretation diverged from the rule.

The money plot is the same shape as §6.1. Triangle = test point; dot = CV mean; dotted lines = 95% CI bounds. The verdict carries the same three options. A small **ABOVE** drift is the typical outcome here because Step 1 refit the tuned GBM on train+val (16,512 rows) while CV's R² estimate was computed on the train slice (12,384 rows) — the deployed GBM sees \~33% more data than CV did, and the test-set R² often lands a hair above the CV CI upper bound for that reason.

Translating R² to USD-RMSE is the version stakeholders will quote. *"R² = 0.81"* is abstract; *"the model is typically off by USD 50,000 on a USD 200,000 home"* is a number HomeValue's deployment council can defend or contest in concrete terms.

Because the random forest performed very similarly (mean 0.804, lower CI 0.794), HomeValue's deployment council should hold the tuned GBM as the **recommended ship pick** while noting the random forest as the close-second runner-up. If a future re-run of nb13 §6 (with a different `random_state`, a different feature set, or fresh data) flips the lower-CI ordering, the verdict can re-open.

**The regression test set is now closed. It will not reopen in this course.**

> **A question that often comes up here:** *"how should HomeValue decide when to retrain the model after deployment?"* Two triggers. (1) **Time-based**: every Q1, retrain on the latest year of housing data and re-run the §5–§6 ceremony to confirm the verdict still holds (same champion model class, CV CI of comparable width, test-set INSIDE). (2) **Drift-based**: if the production median prediction error (USD-MAE) on the most recent 30 days of closed sales drifts beyond about 1.5× the CV MAE, fire a retrain immediately and re-open the ceremony. The discipline is the same one nb14 enforces — never let test-set evidence influence the next model's selection without the protocol being declared first. The deployment council does not have to remember the test-set number forever; it has to remember the rule that fires the next ceremony.

---


### 6.3 Reading the Verdict — What to Do When INSIDE, ABOVE, or BELOW

The money plot pronounces one of three verdicts. The verdict is not a description of model quality alone; it is a **decision rule** that tells you what to do next. The playbooks below are what HomeValue's deployment council and MedScreen's review board need from you in writing, depending on where the test-set triangle lands relative to the dotted CI bounds.

**INSIDE — the test point lands within the CV 95% CI.** Cross-validation was honest: the deployed model performs in the range the CV evidence predicted. **Ship the model.** The deployment memo records (1) the model class and hyperparameters, (2) the CV mean ± 95% CI as the deployment forecast, (3) the test-set point as the one observed instance, and (4) the retrain triggers (Q1 time-based plus drift-based on the production error). The test envelope is now closed; the next ceremony fires only when a retrain trigger fires. INSIDE is the verdict you want and the one a well-disciplined CV pipeline produces most of the time.

**ABOVE — the test point lands above the CV CI upper bound.** Two cases live here with very different actions.

A *small* ABOVE drift — a hair above the upper bound — is **expected and routine** on the course's two demo datasets. Step 1's finalize refit gave the deployed model more data than CV measured (80% vs 60% per fold), so CV was modestly underestimating deployment performance and the test point is the corrected estimate. The action is: **ship the model and document the gap** in the deployment memo so the stakeholder understands why the headline number is higher than the CV forecast. The deployment council should read this as good news — the corrected forecast is closer to what the production model will actually do.

A *large* ABOVE drift — more than about two CI widths above the upper bound — is **not** the train+val story; investigate before deploying. Three possible causes, in order of likelihood. (1) The random test-set draw may have been easier than the CV folds, which is rare on California Housing's 4,128-row test set but plausible on MedScreen's 114-patient test set where a single quirky fold can move the point estimate noticeably. (2) Opposite-direction CV contamination — the CV folds may share a noise source the test set lacks (e.g., a derived feature that averages out across folds), depressing CV scores artificially. (3) A data quality issue isolated to the train pool — mis-labeled rows, outliers, drift between collection windows for train vs test. Document the investigation and either deploy with the larger gap explained, or refit and re-open the ceremony with a *fresh* test set.

**BELOW — the test point lands below the CV CI lower bound.** **STOP — do not deploy.** The CV evidence overestimated deployment performance, which means the development pipeline has a problem the ceremony just exposed. Three diagnoses in order of likelihood:

1. **Leakage during development.** Some feature engineering or preprocessing happened *before* the CV split, leaking information from the validation folds into the training folds. Audit every pipeline step: target encoding, scaling, imputation, derived features. The rule is *"everything fits inside the fold, nothing fits outside it."* If you find leakage, fix the pipeline, refit, and **re-open the ceremony with a NEW held-out test set** — the old test set is now contaminated by the verdict you just observed and cannot give an honest second number.
2. **Distribution shift.** The test set differs systematically from train + val — different time period, different sampling frame, different label noise. Compare summary statistics across the three splits feature by feature. If shift is confirmed, the right response is *not* to retrain on the test set; it is to refresh the development data so train, val, and test all reflect the deployment distribution, then re-run the entire chain from nb09 forward.
3. **Insufficient training signal.** Too few examples per class on classification, too few rows in a key region of feature space on regression. The fix is more data, not a different model.

The hardest rule under BELOW is the **test-set-is-spent** principle: opening the test set once consumes its credibility, regardless of the verdict. If the verdict is BELOW and you diagnose, refit, and want to score again, the second score needs a **new** test set held out from the start. Reusing the same test set after a BELOW would turn the test envelope into a validation set — which is exactly the failure mode §6's entire ceremony exists to prevent.

> **A question that often comes up here:** *"if a BELOW verdict means I can't reuse the test set, am I stuck waiting for new data to retry?"* Often yes — and that is the right discipline. In professional practice the team holds out a third partition (a *gold* test set) at the very start of the project that is sealed until the ceremony, AND keeps a *deployment monitoring* slice that is sampled fresh from production and never used for selection. If a BELOW verdict fires, the gold test set diagnoses what the original test set already showed; the deployment monitoring slice catches drift after release. The test-set discipline is not about being precious with data — it is about preserving the one mechanism that can give an unbiased estimate of deployment performance, which is the only number stakeholders will treat as binding.

---


## 7. Moving Ahead — Keeping the Model Honest After You Ship It

The ceremony in §6 was a one-time event: you opened the locked test set, wrote down the verdict, and closed it for good. But in real life the model keeps making predictions long after that day — new patients arrive for MedScreen, new properties get listed for HomeValue — and the world the model lives in keeps changing. A deployed model **decays** quietly. The accuracy you measured today is not guaranteed to hold next quarter, and the only way to know is to **watch the model in production**.

This section is about that watching. §6 named *which model ships*. §7 answers two follow-up questions every stakeholder will eventually ask: *"how do I know the model is still working?"* and *"when do I do something about it?"*

**The production world can change in two very different ways**, and each calls for a different response. We will call them **Silent decay** (changes that hit *without* announcement — only monitoring catches them) and **Front-door arrivals** (changes that show up *with* notice — your job is a disciplined decision, not detection).

**Silent decay — the quiet quality slip.** Three failure modes that hit a deployed model without warning. The model keeps returning predictions, nothing crashes, but quality slips. These are invisible unless you set up alerts.

1. **Data drift** *(also called covariate shift).* The inputs change — same column names, different values. MedScreen buys new microscopes that measure cell size slightly differently. HomeValue's `MedInc` numbers shift as inflation pushes incomes up. The model never sees the change explicitly; its predictions just get less accurate.

2. **Concept drift.** The *relationship* between inputs and the answer changes. A neighborhood gentrifies and high incomes no longer mean what they meant a few years ago. A new biopsy staining technique exposes a tumor marker the model never trained on, so the old features stop carrying the predictive signal they used to. Same numbers, different meaning.

3. **Population shift.** The people (or properties) the model is applied to change. MedScreen extends screening eligibility from women 50+ to women 40+. HomeValue is asked to price vacation rentals — a property type the training data did not include. The model is now being used on a population it was never trained on.

**Front-door arrivals — the announced change.** The other category is different in tone. Instead of quality slipping in the dark, *more signal* arrives in plain view: a NEW feature becomes available, and somebody tells you about it. Six months after deployment, MedScreen's lab adds a new tissue-marker test to every biopsy panel, or HomeValue gets access to a new census variable nobody had access to before. This is a *good* problem — extra information showing up, not quality dropping — but the response is still disciplined, because the wrong decision can hurt you.

The right response depends on what kind of new feature it is:

- **One new column on the same kind of row** — for example, a new lab measurement attached to each biopsy, or one extra census variable on each property. **Plug the new column into the existing pipeline and refit the same model class with the bigger feature set.** Only put the new version into production if its 5-fold CV 95% CI is at-or-above the deployed model's.

- **A feature that changes the kind of data you have** — for example, a measurement that needs entirely new preprocessing (a new encoder, a new way of handling missing values), or one that suggests a fundamentally different model family would now work. **Re-open the nb14 pipeline (§§4–6) with the new feature included from the start** — a fresh candidate roster, fresh cross-validation, a new champion memo, and a new ceremony to open a fresh test set.

**Three things to check before plugging *any* new feature in**, regardless of which response you choose:

- **Could the new feature be leaking?** A *leaky* feature is one that secretly contains information about the answer. (Example: a "doctor's clinical note" field that was filled in *after* the biopsy result was known will look incredibly predictive — because the note literally summarizes the answer.) The four-method importance heatmap from nb12 §7 is the right diagnostic: if a brand-new feature suddenly dominates all four importance methods and the training error drops dramatically, treat it as a leakage suspect until you can prove it isn't. Gradient boosting (nb13) is especially vulnerable here — its very design hunts for the most informative feature first.

- **Does the new feature actually help by a meaningful margin?** Apply the same CI-overlap rule from §5. The new model with the extra feature only ships if its cross-validation CI sits CI-clear above the deployed model's. If the two CIs overlap, the simpler model wins by parsimony — you do not deploy more complexity unless the evidence demands it.

- **What happens in production when the new feature is missing?** Real-world rows will sometimes lack the new column (the new lab test was not ordered for this patient; the new census release is not yet available for this tract). Decide in advance: do you fill in a sensible default (impute), fall back to the deployed model that did not need the feature, or refuse to predict on those rows? Make the call before you ship.

**Most of the rest of §7 is about catching Silent decay**, because that is the harder problem — you don't get a phone call. The next subsection (§7.1) lays out the *when to act* policy: an escalation ladder of three possible responses, ranked from cheapest to most expensive. §7.2 and §7.3 give the per-case monitoring checklists; §7.4 walks one drift-detection technique in code; §7.5 collapses everything onto a one-page monitoring card. The Front-door-arrival workflow above and the Silent-decay workflow share the same escalation framework, which §7.1 introduces next.

---

### 7.1 Three Things You Can Do — Re-evaluate, Re-train, Rebuild

When monitoring spots something off, you have three possible actions. They come at very different costs, so the rule is: **always reach for the cheapest one that fixes the problem.** Think of it as a three-step ladder where you start at the bottom and only climb if the lower step does not solve it.

1. **Step 1 — Re-evaluate.** *Low cost — a few minutes against a fresh CSV.* Take the exact model already running in production. Score it on the most recent batch of new data (where you already know the right answers). No refitting, no model changes — just *"is the model that has been running still accurate on what just happened?"* Fires on routine monthly or quarterly checks, or whenever monitoring flags something that might be drift.

2. **Step 2 — Re-train.** *Medium cost — same compute as the original training run.* Same model class, same algorithm, same hyperparameters — but **refit on fresh data** that includes the months you have collected since deployment. Compare the refit's 5-fold CV 95% CI to the deployed model's. Only put the refit into production if its CV CI is at-or-above the deployed model's. Fires when Step 1 confirms a real drop, when the data window has grown enough to matter, or when the deployed model has slipped below its original CV-CI lower bound for **two months in a row**.

3. **Step 3 — Rebuild.** *High cost — roughly the same effort as running nb14 from scratch.* Re-open the full nb14 pipeline (§§4–6): a fresh candidate roster (maybe with new model families nobody tried originally), fresh cross-validation, a new champion memo, and a new ceremony to open a fresh test set. Fires when Step 2's refit failed to recover the deployed CI, OR when something fundamental changed (a new feature is now available, the business problem itself shifted, the deployed model's lift over the Week-2 baseline has narrowed too much).

**The escalation rule is mechanical.** Always start at Step 1. If Step 1 confirms drift, escalate to Step 2. If Step 2's refit cannot match the deployed CV CI, only then escalate to Step 3. The reason for the cheapest-first discipline is simple: **Step 3 is essentially a full re-do of nb14**. It costs real time and real money. Make stakeholders sign off before you go up that step.

---

### 7.2 MedScreen Monitoring Plan — What to Track for the Classification Model

For the State Health Department's screening tool, MedScreen ships `LogisticRegression(C=1.0)` (the §6 champion). Logistic regression has one property that matters a lot for monitoring: it gives you **calibrated probabilities** — the model's *"73%"* should genuinely mean *"73% of patients with this score will turn out to be malignant."* That property makes checking the model easier than for some other model families: you can directly compare the probabilities the model is putting out against what actually happened.

The first item in the checklist below uses two tools to make that check concrete:

- The **reliability diagram** is a chart that compares *"the model said 70%"* against *"how often the answer actually was malignant when the model said 70%."* A well-calibrated model's curve hugs the 45° diagonal — predicted 70%, observed 70%.
- The **Brier score** is a single number that summarizes the reliability diagram — *"on average, how far off were the predicted probabilities from the actual outcomes?"* Smaller is better, and you compare it against the Brier score the model had on the §5 CV folds.

The other priority on a screening tool is **recall** (also called sensitivity) — what fraction of actual malignancies the model catches. A screening model that misses real malignancies is much worse than one that occasionally over-flags a benign case, so recall is the number to watch most closely.

**MedScreen monitoring checklist** (five items, each with cadence, alert threshold, and the right escalation step):

1. **Reliability diagram + Brier score** *(on the last 200 confirmed biopsies)*
   - **How often:** monthly
   - **Flag when:** the Brier score climbs above 1.5× the value it had on the §5 CV folds, OR the reliability curve no longer hugs the diagonal
   - **Action:** Step 1 (Re-evaluate)

2. **Recall** *(at the threshold currently used to flag a biopsy for confirmatory testing)*
   - **How often:** monthly
   - **Flag when:** recall drops below 0.92 over a rolling 90-day window
   - **Action:** Step 2 (Re-train), refitting on the most recent 24 months of biopsy data

3. **Input drift** *(on the top-3 features the model relies on — `worst concave points`, `worst perimeter`, `worst radius`)*
   - **How often:** quarterly
   - **Flag when:** a drift test (§7.4 walks the KS option and points at alternatives) crosses your alert threshold
   - **Action:** Step 1 first; if drift is confirmed, escalate to Step 2

4. **Class prevalence** *(how often the answer is actually malignant)*
   - **How often:** quarterly
   - **Flag when:** the malignant rate has shifted by more than ±20% from what the training data showed
   - **Action:** Step 1; you may also need to move the threshold to keep the model useful at the new base rate

5. **Annual clinical review** *(by the State Health Department's review board)*
   - **How often:** yearly
   - **Flag when:** new screening guideline, new imaging equipment, OR a new tissue-marker test becomes available
   - **Action:** Step 3 (Rebuild) — or Step 2 Re-train if the new marker is just an extra column (see §7's opening on new features)

**Who owns this?** The data steward who maintains the biopsy data pipeline runs the monthly checks. The oncologist co-signs every quarterly review. Both sign off jointly before any Step-2 or Step-3 action goes live. In practice, the monitoring checklist above becomes a **one-page report** the State Health Department's review board reads at the start of every quarterly meeting.

---

### 7.3 HomeValue Monitoring Plan — What to Track for the Regression Model

For HomeValue Analytics' price-prediction tool, the champion is `GradientBoostingRegressor(learning_rate=0.2, n_estimators=200, max_features=0.5, max_depth=3)` — the tuned GBM from nb13 §6, picked in §6.2 as a *modest winner* under the dominance tiebreaker. Unlike MedScreen's logistic regression, gradient boosting does not output probabilities — it outputs **dollar amounts**. So the monitoring story shifts to two questions: *how big are the prediction errors right now*, and *do those errors get worse for certain kinds of properties* (for example, the high-end properties where the model already showed it tends to underestimate during the §5 diagnostics).

**HomeValue monitoring checklist** (six items, each with cadence, alert threshold, and the right escalation step):

1. **Average and typical prediction errors (MAE / RMSE)** *(on last month's appraisals)*
   - **How often:** monthly
   - **Flag when:** RMSE climbs above USD 70,000 (compared to the \~USD 50,000 baseline from §5's cross-validation) over a rolling 90-day window
   - **Action:** Step 1 (Re-evaluate)

2. **Errors broken out by price tier** *(split properties into five equal-sized price bins — quintiles — and compute RMSE in each)*
   - **How often:** quarterly
   - **Flag when:** top-quintile (the priciest 20% of properties) RMSE climbs above USD 110,000, compared to its \~USD 85,000 baseline
   - **Action:** Step 2 (Re-train) on the most recent 36 months of sales

3. **Errors by location** *(group properties into a coarse latitude/longitude grid; compute average error in each cell with more than 100 properties)*
   - **How often:** quarterly
   - **Flag when:** any cell's average error has shifted by more than half a standard deviation from what it was at training time
   - **Action:** Step 1 on that cell; consider a region-specific refit if a neighborhood has genuinely changed

4. **Input drift** *(on the most important features — `MedInc`, `AveOccup`, `Population`)*
   - **How often:** monthly
   - **Flag when:** a drift test (§7.4 walks the KS option and points at alternatives) crosses your alert threshold
   - **Action:** Step 1; if confirmed, escalate to Step 2

5. **Major outside events**
   - **How often:** whenever one happens (event-driven)
   - **Flag when:** the Federal Reserve changes interest rates by more than 50 basis points (half a percentage point), OR the US Census releases a new wave of demographic data, OR a major housing-policy change takes effect
   - **Action:** Step 1 immediately, and flag the deployment council that the model may need attention

6. **Annual model audit** *(by the deployment council)*
   - **How often:** yearly
   - **Flag when:** the random forest's advantage over the simple OLS baseline has narrowed to fewer than 10 R² points, OR a new dataset becomes available with previously-unavailable features
   - **Action:** Step 3 (Rebuild) — or Step 2 Re-train if the new feature is just an extra column (see §7's opening on new features)

**Who owns this?** The pricing-analytics team runs the monthly checks. The deployment council co-signs the quarterly review. The "major outside events" item is set up to fire automatically off public-data feeds (Federal Reserve announcements, Census releases, and so on) — the data steward keeps that script running. Same one-page-report discipline as MedScreen.

---

### 7.4 Drift Detection in Code — One Approach (and a Note That Others Exist)

The monitoring checklists above kept using the phrase *"a drift test"* without explaining what one looks like in code. This subsection walks **one example** — the **Kolmogorov-Smirnov test**, usually abbreviated as the **KS test**. It is one of the simplest drift detectors and a good starting point for the course. **It is not the only option**, and depending on your data and your business setting you may want to research alternatives (see the list at the end of this subsection).

**What does a KS test do, in one sentence?** Take the distribution of a feature in your training data, take the distribution of the same feature in your recent production data, and measure how far apart those two distributions are. If the two distributions look basically the same shape, the test returns a small number (no drift). If they look meaningfully different — shifted left, shifted right, narrower, wider — the test returns a larger number (drift detected). The "KS statistic" itself is a number between 0 and 1 — bigger means more different. In production you would pick a threshold (the monitoring checklists above used 0.10 for HomeValue's features and 0.15 for MedScreen's, but those are starting values you tune for your case after observing some real production data) and treat any feature where the KS statistic exceeds the threshold as an alert.

The code cell below runs this on one important feature from each business case. For MedScreen we use `worst perimeter` (one of the top features the logistic regression relies on). For HomeValue we use `MedInc` (the single most important feature by permutation importance from nb12). For each case we run **three scenarios**: a *no-shift* baseline (the training distribution compared to itself — must always come back `✓ OK`), a *modest* simulated shift that should fall under the 0.10 alert threshold, and a *bigger* simulated shift that should clearly cross it. Then we read the KS test results side by side and visualize the bigger-shift production distributions against training.

> **Other drift-detection options worth researching when you build this for a real system.** The KS test is just one tool in a much wider area called *data-drift detection*. Other widely used approaches include:
>
> - **Population Stability Index (PSI)** — heavily used in banking and credit-risk modeling.
> - **Chi-squared test** — the natural choice for *categorical* features (where the KS test does not work directly).
> - **Wasserstein distance** (also called *Earth Mover's Distance*) — measures the *size* of a shift, not just whether one happened.
> - **Jensen-Shannon divergence** — an information-theoretic measure of how different two distributions are.
> - **Plain summary-statistic comparisons** — comparing means, standard deviations, percentiles side-by-side over time. Often the simplest dashboard is the most useful.
>
> Different industries default to different tools — finance leans on PSI, manufacturing reaches for Wasserstein when small tolerance shifts matter, healthcare often uses chi-squared on categorical lab-result fields. The right choice depends on your data and what kind of shift you most want to catch. A quick search for *"data drift detection methods"* will turn up active research and several open-source libraries (`evidently`, `alibi-detect`, `nannyml` are three popular ones) that bundle multiple tests into a single workflow. None of these are required for the work this notebook covers — they are starter pointers for when your group is ready to build monitoring for a real deployed model.

> 💡 **Gemini Prompt:** "Write a Python cell that runs a Kolmogorov-Smirnov drift test on one feature from each business case. Use `scipy.stats.ks_2samp`. Define `drift_check(name, train_values, prod_values, alert_threshold=0.10)` that returns a formatted string starting with `'🚨 DRIFT'` when the KS statistic exceeds `alert_threshold` and `'✓ OK'` otherwise, followed by the feature name (left-padded to 40 chars), the KS statistic to 3 decimals, and the p-value to 3 significant figures.
>
> For the **classification** case, pull `X_train_clf['worst perimeter']` and create three production scenarios drawn from one `np.random.RandomState(RANDOM_SEED)`: (a) the training values unchanged ('no shift'), (b) a *small sensor drift* adding `normal(0, 5, n)` noise, and (c) a *bigger sensor drift* adding `normal(0, 20, n)` noise — both drawn sequentially from the same RNG so the runs are reproducible. Print all three rows under a `=== CLASSIFICATION (MedScreen) ===` header.
>
> For the **regression** case, pull `X_train_reg['MedInc']` and create three scenarios: (a) no shift, (b) a 10% upward income shift (`* 1.10`), and (c) a 20% upward shift (`* 1.20`). Print all three rows under a `=== REGRESSION (HomeValue) ===` header.
>
> Then build a 1×2 figure with `plt.subplots(1, 2, figsize=(15, 5))`. The left panel overlays the MedScreen training distribution against the *std=20* production distribution. The right panel overlays the HomeValue training distribution against the *20%* production distribution. Use `CLF_COLOR` / `REG_COLOR` for training, `RED` for the drifted production. Add 30-bin histograms with `alpha=0.5`, titles, axis labels, legends, gridlines, and a `fig.suptitle` describing what the figure shows."
>
> **After running, verify:**
> - [ ] Six labeled rows total (three per case), printed under their respective `=== CASE ===` headers
> - [ ] Both "no shift" rows show `KS=0.000` and `✓ OK`
> - [ ] The modest-shift rows in both cases come in *below* the 0.10 threshold and show `✓ OK`
> - [ ] The bigger-shift rows in both cases come in *above* the 0.10 threshold and show `🚨 DRIFT`
> - [ ] The 1×2 figure renders with the bigger-shift production histogram visibly offset from training in both panels
> - [ ] The HomeValue p-values are dramatically smaller than the MedScreen p-values, even when the KS statistics are similar — that asymmetry is one of the lessons in the Reading-the-output below

In [ ]:
# §7.4 — Drift detection demo: KS (Kolmogorov-Smirnov) test on one feature per business case.
#
# What the KS test returns:
#   - `stat` is between 0 and 1. Bigger = the two distributions look more different.
#   - `p` is a p-value. Smaller means the two distributions are more confidently different.
# We use only `stat` for our alert decision (a stat above the alert threshold = DRIFT).
# The p-value is informative but should NOT drive the alert by itself — see the Reading
# below for why large sample sizes shrink p-values regardless of how big the shift is.

from scipy.stats import ks_2samp

def drift_check(name, train_values, prod_values, alert_threshold=0.10):
    stat, p = ks_2samp(train_values, prod_values)
    flag = '🚨 DRIFT' if stat > alert_threshold else '✓ OK'
    return f'{flag}  {name:<40} KS={stat:.3f}  p={p:.3g}'

# --- Classification (MedScreen): 'worst perimeter' is a top feature for the LogReg champion ---
clf_train_feat  = X_train_clf['worst perimeter']
rng = np.random.RandomState(RANDOM_SEED)
clf_prod_modest = clf_train_feat + rng.normal(0,  5, len(clf_train_feat))   # small sensor noise
clf_prod_bigger = clf_train_feat + rng.normal(0, 20, len(clf_train_feat))   # heavier sensor noise

print('=== CLASSIFICATION (MedScreen) ===')
print(drift_check('worst perimeter (no shift)',            clf_train_feat, clf_train_feat))
print(drift_check('worst perimeter (small sensor drift)',  clf_train_feat, clf_prod_modest))
print(drift_check('worst perimeter (bigger sensor drift)', clf_train_feat, clf_prod_bigger))

# --- Regression (HomeValue): 'MedInc' is the most important feature by permutation importance from nb12 ---
reg_train_feat  = X_train_reg['MedInc']
reg_prod_modest = reg_train_feat * 1.10    # 10% upward income shift (inflation / gentrification)
reg_prod_bigger = reg_train_feat * 1.20    # 20% upward income shift

print('\n=== REGRESSION (HomeValue) ===')
print(drift_check('MedInc (no shift)',                 reg_train_feat, reg_train_feat))
print(drift_check('MedInc (10% upward shift)',         reg_train_feat, reg_prod_modest))
print(drift_check('MedInc (20% upward shift)',         reg_train_feat, reg_prod_bigger))

# --- Visualize the bigger-shift scenarios side by side ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].hist(clf_train_feat,  bins=30, alpha=0.5, color=CLF_COLOR, label='Training data')
axes[0].hist(clf_prod_bigger, bins=30, alpha=0.5, color=RED,       label='"Production" — bigger sensor drift')
axes[0].set_title("MedScreen — 'worst perimeter' (training vs bigger-shift production)", fontsize=12, fontweight='bold')
axes[0].set_xlabel('worst perimeter'); axes[0].set_ylabel('count'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].hist(reg_train_feat,  bins=30, alpha=0.5, color=REG_COLOR, label='Training data')
axes[1].hist(reg_prod_bigger, bins=30, alpha=0.5, color=RED,       label='"Production" — 20% income shift')
axes[1].set_title("HomeValue — 'MedInc' (training vs 20%-shift production)", fontsize=12, fontweight='bold')
axes[1].set_xlabel('MedInc (10K USD units)'); axes[1].set_ylabel('count'); axes[1].legend(); axes[1].grid(alpha=0.3)

fig.suptitle('Drift detection in code — the bigger shifts that crossed the 0.10 alert threshold', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

**Reading the output:**

Six rows total, three per case. Two clear patterns to read off.

**The "no shift" rows are the sanity check.** Both come back at `KS=0.000` with `p=1` and the `✓ OK` flag. That has to be true — comparing a distribution to itself can never look different, so any failure here would be a bug in the code, not a finding. Always include this row in monitoring scripts so a regression in the *test infrastructure itself* shows up immediately, before you stake any deployment decision on the test's other rows.

**MedScreen — small samples, the noise floor of drift detection.** The "small sensor drift" row (random noise with std 5, on top of `worst perimeter` whose own std is about 33) lands at **KS \~ 0.053** with `p ~ 0.73` — well below the 0.10 alert threshold, so `✓ OK`. The "bigger sensor drift" row (random noise with std 20) lands at **KS \~ 0.106** with `p ~ 0.045` — just above the threshold, so `🚨 DRIFT`. With only 341 training patients, KS values move noticeably between scenarios, and the bar to cross the threshold sits well within what a realistic microscope recalibration could produce. The lesson for the State Health Department's review board: small sensor changes blur into measurement noise; a microscope swap large enough to perceptibly shift the readings is exactly what the test should catch — but on a 341-patient pool you need a real shift to see it.

**HomeValue — what large sample sizes do to p-values.** The "10% upward shift" row lands at **KS \~ 0.090** with a p-value of **about 4 × 10⁻⁴⁴** — `✓ OK`. The "20% upward shift" row lands at **KS \~ 0.168** with a p-value near **7 × 10⁻¹⁵⁴** — `🚨 DRIFT`. Both p-values are vanishingly small even though the first scenario falls *below* the alert threshold. The KS statistic itself measures the *size* of the shift; the p-value answers a different question — *"could this shift be pure sampling noise?"* — and on 12,384 tracts the answer is always *"no, with extreme confidence,"* regardless of whether the shift is operationally meaningful. **That is why the alert threshold is set on the KS statistic, not on the p-value.** A 10% income shift across the whole state is real (it shows up unambiguously in the data) but it is not yet big enough to demand an immediate model re-train; a 20% shift is. The business judgment lives in choosing the threshold on the magnitude metric, not on the certainty metric.

> **A question that often comes up here:** *"if the p-value is essentially zero, shouldn't that always trigger an alert?"* No — and that intuition is the most common mistake when teams first set up drift monitoring. p-values are about *evidence that the distributions differ*, not about *how big the difference is*. With enough samples, any non-identical pair of distributions will produce a tiny p-value — even a shift too small to care about operationally. **Always alert on the KS statistic's magnitude** (or the equivalent magnitude statistic for whichever drift test you choose); use the p-value only as a sanity check that the test ran cleanly.

> **A question that often comes up here:** *"how do the two business cases compare?"* The MedScreen rows show how hard it is to detect small drifts on small datasets — you need a bigger physical change before the test sees it. The HomeValue rows show the opposite — on a 12,000-tract training set the test sees every shift with overwhelming statistical confidence, so the threshold has to do the work of separating *real* from *operationally meaningful*. Both lessons matter when you set the thresholds on your own monitoring card.

---

### 7.5 Putting It All on One Page — The Monitoring Card

Each model you deploy gets exactly **one page** of monitoring documentation. Four sections:

1. **Which model is running?** Model class, hyperparameters, and the cross-validation 95% CI from §5. *(For MedScreen: `LogReg(C=1.0)` inside a `StandardScaler` pipeline, CV ROC-AUC \~0.994 [\~0.986, \~1.001]. For HomeValue: tuned `GradientBoostingRegressor(learning_rate=0.2, n_estimators=200, max_features=0.5, max_depth=3)`, CV R² \~0.812 [\~0.798, \~0.826].)*
2. **What did the test set say, and when?** The INSIDE / ABOVE / BELOW verdict from §6, the test-set number, and the date the envelope was opened. One line that proves the model was picked before anyone looked at the test set.
3. **What are we tracking?** The condensed monitoring checklist from §7.2 (for MedScreen) or §7.3 (for HomeValue) — just the actionable rows.
4. **Who acts, and how?** Who runs the monthly checks, who co-signs the quarterly review, and who has the authority to escalate to Step 2 or Step 3.

The same one-page template works for every model your team ships across your career. Draft it once at deployment time and it is ready every quarter when a stakeholder asks *"how do we know the model is still working?"* It is also the document you hand to any new analyst inheriting the model — one page they can read in five minutes, instead of a meeting they need to schedule.

> **A question that often comes up here:** *"how much drift is 'enough' to act?"* The single best heuristic the course gives you is this: **act when the production metric drops below the lower bound of the §5 cross-validation CI for two consecutive monitoring periods.** One bad month, by itself, might just be ordinary fold-to-fold noise — the kind you saw moving around between CV folds in §5's dot plot. But two consecutive bad months in a row is signal, not noise, and that is when you start Step 1 (re-evaluate). The drift-test thresholds in the monitoring checklists (0.10 / 0.15 in the KS tests above) are conservative starting values that work fine before you have any production history; once you have six months of monitoring data, you can tune them up or down based on what actually counts as "unusual" in your specific business context.

---

## 8. Wrap-Up — Key Takeaways

**What landed today:**

1. **The selection protocol is a structured ceremony, not a spreadsheet exercise.** Every reference model introduced from nb01-nb13 enters the roster (7 classification candidates, 8 regression candidates), with **the exact hyperparameters its source notebook shipped**. Declared in advance, evaluated on identical CV folds, with one declared primary metric and supporting metrics for tie-breaking.
2. **The CV-CI dot plot is the central artifact.** CIs that overlap → simpler model wins. CIs that do not overlap → the top model has earned displacement.
3. **The champion is justified in stakeholder language.** Three sentences: what model are we shipping, what's the CV confidence interval on its performance, and what would trigger a revisit.
4. **The locked test set opens exactly once per case.** Two ceremonies in this notebook is a teaching demo; in any single real business analysis, the test set opens once and only once.
5. **The verdict is INSIDE / ABOVE / BELOW** — not a number to debate, but a categorical pronouncement against the CV CI.
6. **Monitoring is the second half of the job.** §7 builds the one-page monitoring card every deployed model needs: *what to track*, *how often*, *what counts as an alert*, and *which step (Re-evaluate, Re-train, or Rebuild) you escalate to when an alert fires*. Drafting that card at deployment time means you have an answer ready every quarter when a stakeholder asks whether the model is still working.

**The two champions, named:**

- **Classification (MedScreen)** — `LogisticRegression(C=1.0)`. CIs of the top four candidates (LogReg, LogReg L1, Random Forest, tuned GBM) overlap heavily, so the CI-overlap rule + parsimony picks the simplest linear baseline. Same verdict nb12 §9 and nb13 §7 reached under the same rule.
- **Regression (HomeValue)** — `GradientBoostingRegressor(learning_rate=0.2, n_estimators=200, max_features=0.5, max_depth=3)` — the tuned GBM from nb13 §6. It dominates the runner-up Random Forest on mean R² (0.812 vs 0.804), mean RMSE (0.499 vs 0.509), mean MAE (\~ tied at 0.336), AND CV CI lower bound (0.798 vs 0.794). The upper-end CI overlap does not earn a statistical tie because the GBM's worst-case fold dominates the random forest's worst-case fold — the **dominance tiebreaker** picks the GBM as a **modest winner**. The Random Forest stays on as the close-second runner-up. Same final pick as nb13 §7's analysis.

**Bridge to nb15 — Complex Model + Tuning + Abstract Walkthrough:**

The champions are now committed. nb15 turns the protocol you just ran into a complete deployment package on both demo cases: replicating the linear baseline with 5- or 10-fold CV and a 95% CI, tuning a more complex model with `GridSearchCV` / `RandomizedSearchCV`, applying the CI-overlap rule to decide whether the complex model has earned displacement of the baseline, retraining on the full training fold with `random_state` locked so the fitted Pipeline reproduces cleanly when M4 re-executes the modeling cell. nb15 also covers the required visualizations (hyperparameter-search plot, model-comparison bar with 95% CI, feature importance, plus regression or classification diagnostics) and a 250-word draft-abstract template — and shows how the same discipline drives the parallel Course Case Competition (Kaggle Bank Churn).

> **A question that often comes up at this point:** *"if nb14 already named the champions, why does nb15 exist?"* Because a complete deployment package is more than naming a model. nb15 requires (a) replicating the linear baseline with a 95% CI, (b) running a systematic hyperparameter search, (c) applying the CI-overlap rule, (d) retraining on the full training fold with `random_state` locked so M4 reproduces the same fitted Pipeline, (e) producing every required visualization, and (f) writing a polished 250-word abstract. nb14 picked the champion in two demo cases under CI-overlap + parsimony on a fixed roster from previous notebooks; nb15 turns that protocol into a complete deployment walkthrough you can copy onto your own dataset.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete both PAUSE-AND-DO exercises** — Exercise 1 (pick the classification champion + explain) and Exercise 2 (pick the regression champion + explain).
2. **Run All Cells** — `Runtime → Run all` to execute every cell including the two §6 ceremonies.
3. **Save a Copy** — `File → Save a copy in Drive`, or download as `.ipynb`.
4. **Submit** — upload the `.ipynb` file to the Notebook 14 participation assignment on Brightspace.

### Before Submitting, Check:

- [ ] Both champion-pick code cells set the required variables (`champ_clf_name/_mean/_sd`, `champ_reg_name/_mean/_sd`)
- [ ] Both champion explanations are written in stakeholder language (3 sentences each)
- [ ] Both money plots (test point vs CV CI) render with the verdict color coded
- [ ] You can defend the singleness rule in plain English

### Next Step:

- **Notebook 15** — Complex Model + Tuning + Draft Abstract Walkthrough (Day 15)

---

---

## Bibliography

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2023). *An Introduction to Statistical Learning with Python* — Ch. 5 (Resampling Methods). Springer.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* — Chapters on model assessment and selection.
- Provost, F., & Fawcett, T. (2013). *Data Science for Business* — Chapter on evaluation and decision protocols.
- Cawley, G. C., & Talbot, N. L. C. (2010). "On over-fitting in model selection and subsequent selection bias in performance evaluation." *Journal of Machine Learning Research*, 11, 2079–2107.
- Dietterich, T. G. (1998). "Approximate statistical tests for comparing supervised classification learning algorithms." *Neural Computation*, 10(7), 1895–1923.
- Mitchell, M., Wu, S., Zaldivar, A., Barnes, P., Vasserman, L., Hutchinson, B., Spitzer, E., Raji, I. D., & Gebru, T. (2019). "Model Cards for Model Reporting." *Proceedings of the Conference on Fairness, Accountability, and Transparency (FAT\* '19)*, 220–229.
- Sculley, D., Holt, G., Golovin, D., Davydov, E., Phillips, T., Ebner, D., Chaudhary, V., Young, M., Crespo, J.-F., & Dennison, D. (2015). "Hidden Technical Debt in Machine Learning Systems." *Advances in Neural Information Processing Systems (NeurIPS)*, 28.
- Provost, F., & Fawcett, T. (2013). *Data Science for Business* — Chapter on deployment, monitoring, and the model lifecycle.
- scikit-learn User Guide — [Cross-validation: evaluating estimator performance](https://scikit-learn.org/stable/modules/cross_validation.html)
- scikit-learn User Guide — [Common pitfalls in interpretation and evaluation](https://scikit-learn.org/stable/common_pitfalls.html)

---

<center>

**Thank you!**

</center>